# DIMER Notebook: Zero-Shot Time-Series Forecasting with TimesFM — Can a Foundation Model Beat a Seasonal Rule of Thumb?

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/timesfm-forecasting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/timesfm-forecasting-pipeline/blob/main/tutorials/timesfm_forecasting_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Ftimesfm--3.0--pytorch-ffcc4d?style=flat)](https://huggingface.co/google/timesfm-3.0-pytorch) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Ftimesfm-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/timesfm) [![arXiv](https://img.shields.io/badge/arXiv-2310.10688-b31b1b.svg)](https://arxiv.org/abs/2310.10688) [![Code: Apache-2.0](https://img.shields.io/badge/Code-Apache--2.0-blue.svg)](https://github.com/kurtvalcorza/timesfm-forecasting-pipeline/blob/main/LICENSE) [![Weights: non-commercial](https://img.shields.io/badge/Weights-TimesFM%20Non--Commercial%20v1.0-red.svg)](https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot univariate time-series forecasting over one or many independent series with the pinned TimesFM 3.0 PyTorch checkpoint (non-commercial weights): a median point forecast and model quantiles from the trained decile grid, evaluated chronologically against naive, seasonal-naive and exponential-smoothing baselines with MASE, sMAPE and quantile loss

**This notebook is standalone.** It carries the repository's package (7 modules under `src/timesfm_forecasting/`, at revision `fc4c39fe9818`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `43046b85ec22d584a13f8098c2ed39c889e129c2` (~1323 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds the isolated hash-locked environment (nothing is installed into the kernel, no restart), stages the pinned TimesFM 3.0 checkpoint (1.32 GB safetensors, non-commercial weights licence) and checks its sizes and SHA-256 digests, fetches the digest-pinned Open-Meteo sample (31 KB, no credential), validates it into an input manifest and shows one named refusal, holds out the last 24 hours of each series chronologically, scores naive, seasonal-naive and exponential-smoothing baselines, runs the zero-shot forecast with median and quantile output, records MASE, sMAPE, quantile loss and band coverage beside the baselines without asserting a winner, varies the context length once, forecasts beyond the data, and exports a result bundle that is reloaded with a parity check. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in Section 4, set `BYOD_PATH` to a CSV on the runtime (or leave it empty for the Colab upload dialog), set the three column fields to your file's names, and choose **Run after**. Your data passes through the same validation (every refusal names the file and the rule), split, baselines, forecast, evaluation and export as the sample; the expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and the file stays inside this runtime. BYOD is optional and never part of the default path.

**The question this notebook answers on one real series:** can a pretrained time-series foundation model, given nothing but the recent history, forecast the next day of hourly temperature in three Philippine cities better than repeating yesterday? The answer is measured, recorded and interpreted; it is not assumed either way.

**Zero-shot does not mean untrained.** TimesFM 3.0 was pretrained by Google Research on a large corpus of time series. *Zero-shot* describes this notebook's task: the model forecasts these series without any training on them. Everything the model knows about daily cycles it learned elsewhere.

**The point forecast is the median, and the band is a model statement.** The pipeline exports the model's quantile 0.5 (slot 4 of its nine quantile slots) as `prediction` and the requested deciles as `q0.1` / `q0.5` / `q0.9`; TimesFM 3.0 has no mean head. The q0.1–q0.9 band is what the model's quantile head emits for a nominal 80 % interval; Section 9 measures how often the truth falls inside it, and only that measurement says how to read it. No decision threshold is shipped.

**Weights licence.** The TimesFM 3.0 weights that Section 3 downloads are distributed by Google LLC under the *TimesFM Non-Commercial License v1.0* (`license: other` on the Hub): **non-commercial and non-production use only** — testing, evaluation and research not tied to commercial gain, production deployment or revenue generation — with no redistribution of the model or of derivatives, and a commercial licence required for anything else. Running this notebook for a client deliverable, a paid product, or any end-user-facing or production system is outside that licence. The notebook's code and the pipeline package are Apache-2.0; the sample is CC BY 4.0. If the restriction does not fit your use, the repository's TimesFM 2.5 build (Apache-2.0 weights) is the fallback; see the repository README.

**Model revision status.** The checkpoint is pinned by repository id, immutable commit, file names, byte sizes and the SHA-256 of `config.json` and `model.safetensors`. The commit and the weight digest are the values the Hub served and the pipeline computed on the maintainer's hosted Colab run of 2026-10-06 (recorded in the repository's `docs/release-verification.md`); Section 3 refuses the snapshot on any size or digest mismatch, and Section 12 exports the commit and the digest it verified.

**Learning objectives:** by the end you should be able to (1) explain *history → pretrained model → median and quantile forecast*; (2) identify why the future is held out chronologically and what leakage would look like; (3) compare the model with naive, seasonal-naive and exponential-smoothing baselines on the same held-out rows; (4) interpret MASE, sMAPE, quantile loss and band coverage, and distinguish a model quantile from a calibrated interval; (5) predict and measure the effect of the context length in a controlled activity; and (6) apply the same validation, forecast and export contract to your own CSV and read its refusals.

**This notebook does not demonstrate:** covariates (past-only or known-future regressors; TimesFM 3.0's covariate inputs are not exposed), multivariate joint forecasting, fine-tuning or any training, anomaly detection, imputation as a product feature (the pipeline's interpolation is a reported validation policy, not a model), probability calibration or conformal intervals, rolling-origin backtesting, multivariate variate attention across series (each series is forecast independently), and any claim that one holdout day on three series stands in for your data. The repository exposes none of these.

## Who this notebook is for, and how to use it

**Learner.** Someone who can open a hosted notebook, run cells and read basic Python, and who is new to time-series forecasting or to foundation models for it. No prior forecasting or machine-learning course is assumed: horizon, context, holdout, baseline, MASE and quantile are explained where they first matter and collected in the **Glossary** at the end. No accelerator is required; the default path runs on CPU and uses a GPU automatically when one is present.

**How to use this notebook.** In Colab, choose any runtime (CPU is fine), then **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried package code, and the checkpoint staging — labelled as such and collapsed; you may run them without studying their implementation. The learning path starts in Section 4. The form fields (`# @param`) in Sections 4 and 10 are the only values meant to be edited, and the defaults reproduce the canonical path. Each stage states its question, asks you to **predict** before it runs, and ends with **What to notice**; collapsible **Check your reasoning** blocks give worked guidance after you have answered. **Troubleshooting** and the **Glossary** are at the end. Nothing you write in the conclusion scaffold is a required submission.

**By the end you should be able to** (1) explain the task as *history → pretrained model → median and quantile forecast*; (2) identify why the future must be held out chronologically; (3) compare a foundation model with naive, seasonal-naive and exponential-smoothing baselines on the same held-out rows; (4) interpret MASE, sMAPE, quantile loss and band coverage without reading a model quantile as a guarantee; (5) predict and then measure the effect of one change (the context length); and (6) apply the same validation, forecast and export contract to your own CSV.

## Roadmap and the task contract

**Roadmap.** 4 obtain the sample (or your CSV) → 5 validate into an input manifest, see a refusal → 6 split chronologically → 7 three baselines → 8 **zero-shot forecast** with point and quantile output → 9 evaluate beside the baselines (verdicts recorded, never asserted) → 10 **change one thing: the context length** → 11 forecast beyond the data → 12 export and reload → conclude. A fast path is the default path: nothing is optional until Section 10, and Section 10 is bounded.

**Input → Model → Output.** *Input:* a long-format table — timestamp, numeric value, optional series id — sampled at one interval. *Model:* TimesFM 3.0 (a stacked mixing transformer pretrained on time series; the Hub lists 330.7M parameters; weights under a non-commercial, non-production licence) reading the most recent `CONTEXT_LENGTH` points of each series, with nothing trained here. *Output:* for each series and each of `HORIZON` future steps, a median point forecast and the model quantiles q0.1 / q0.5 / q0.9; beside it, the same metrics for three history-only baselines, an input manifest, an evaluation report and a result bundle with provenance.

**Three ideas to hold on to.** *Zero-shot* describes this notebook's task — no training on these series — not the model's ignorance of weather in general. *Baselines come first*: a seasonal rule of thumb is the bar, and the notebook records whether the model clears it rather than assuming it. *Quantiles are the model's statements*: the band's measured coverage on the holdout is what tells you how to read it.

## Prerequisites

- **Learner:** basic Python and Colab familiarity; no prior forecasting experience. New terms (horizon, context, holdout, baseline, MASE, quantile) are explained where they are first used and collected in the Glossary.
- **Weights licence (read before Section 3):** the pinned `google/timesfm-3.0-pytorch` weights are under the *TimesFM Non-Commercial License v1.0* — **non-commercial and non-production use only**, no redistribution of the model or of derivatives, a commercial licence from Google LLC required for any other use. Downloading them in Section 3 means accepting those terms. The pipeline code is Apache-2.0 and the sample is CC BY 4.0; the licence text is in the checkpoint repository's `LICENSE` file.
- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab, Kaggle or Linux Jupyter. Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter, and a Windows or macOS kernel is not supported (Section 1 stops with that message). The default path runs on CPU in float32 and uses CUDA automatically when available. The locked install (PyTorch 2.14.0 with its CUDA libraries) and the 1.32 GB checkpoint are the large downloads. No hosted run is recorded yet, so no runtime figure is given; the model stages on this sample are three series of 312 points at a 24-step horizon, which is small for a 330M-parameter model on CPU (an expectation, not a measurement).
- **Data contract:** one CSV, long format — a timestamp column (ISO-8601 or any pandas-parseable format), a finite numeric value column, and optionally a series-id column; one sampling interval per file (fixed, such as hourly or daily, or a calendar interval such as monthly); no duplicate timestamps within a series; at least `HORIZON + 16` rows per series (40 with the default horizon) and at least 3 rows to infer the interval; at most 1,000 series and 2,000,000 rows; histories longer than 15,360 points (the model's context limit) are truncated to their most recent points with a report. Missing values and skipped grid points are refused by default and may instead be interpolated / filled with a report (`NAN_POLICY`, `GAP_POLICY`; at most 20 % of a series may be interpolated). Section 5 checks all of this before any model call and names the file and the rule it refuses.
- **Validation is structural, not semantic:** nothing checks that the values mean what you think, that the series is stationary, or that the season length you set is the data's real period; a wrong `SEASON_LENGTH` makes the seasonal baseline and the MASE scale wrong without any error.
- **Privacy:** do not upload confidential or restricted data to a hosted runtime unless you are authorised to process it there — operational telemetry or sales figures may be exactly that. The default path uploads nothing.
- **External access:** the Hugging Face Hub, to fetch the pinned `google/timesfm-3.0-pytorch` files (~1323 MB in total; non-commercial weights licence, see above) at the immutable revision `43046b85ec22d584a13f8098c2ed39c889e129c2` (the commit the Hub served on the maintainer's hosted run of 2026-10-06; the weight digest observed on that run is pinned in the manifest), and `raw.githubusercontent.com` for the pinned sample (`openmeteo_ph_hourly_temperature.csv`, 31,275 bytes, SHA-256 `74163ee6…`, the copy committed under `examples/sample-data/` of this repository at commit `e47259ae75b9…`; Open-Meteo, CC BY 4.0), refused on any mismatch before it is read. The sample is data, not code: nothing is installed from this repository and no credentials are required The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 52 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 52 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'timesfm==3.0.2',
    'torch==2.14.0',
    'numpy==2.5.3',
    'pandas==3.0.5',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'matplotlib==3.10.8',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = 'ed3a5af7014579815c2472f6d4f2d0c7cef0bb71651da52f1c1e751313a20b7a'
LOCKED_PACKAGES = 52
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile tutorials/requirements-colab.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
contourpy==1.4.0 \
    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \
    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \
    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \
    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \
    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \
    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \
    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \
    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \
    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \
    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \
    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \
    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \
    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \
    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \
    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \
    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \
    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \
    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \
    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \
    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \
    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \
    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \
    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \
    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \
    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \
    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \
    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \
    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \
    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \
    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \
    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \
    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \
    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \
    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \
    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \
    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \
    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \
    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \
    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \
    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \
    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \
    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \
    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \
    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \
    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \
    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \
    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \
    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \
    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \
    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \
    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \
    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \
    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \
    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \
    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \
    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \
    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \
    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \
    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \
    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \
    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \
    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \
    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \
    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \
    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \
    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \
    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \
    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \
    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \
    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707
    # via matplotlib
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
cycler==0.12.1 \
    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \
    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c
    # via matplotlib
filelock==4.0.12 \
    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \
    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c
    # via
    #   huggingface-hub
    #   torch
fonttools==4.66.1 \
    --hash=sha256:058cd823b80bac59e64dfad9e3b6fcd677852f9a3804971bbf6b48cc611e785c \
    --hash=sha256:05aeb146451f37289f782c3c861f3d0f4b86c2dd2e4620b46683544c7406640e \
    --hash=sha256:05c0fff6b4a5d872ed89cab2c4f81060b86ace263903eb4e8d0edcac47a60dfa \
    --hash=sha256:08d8956e3ec990c75230d92f1630b215e8f3738c83a003421c22b31ebfd0ce15 \
    --hash=sha256:09ae73bd219e1245debd8376077a0fa6e03175e255c4f51bae5f6a271bfe384a \
    --hash=sha256:0dc6fd99cb8c30941036308b148da9432640442a6f26f36d71dad9be24cbd0e9 \
    --hash=sha256:1801fdad5600118327171e0e8aa79f7cc48831dd55ab36998c9de03bd5ffe6cd \
    --hash=sha256:261d8dc95845e751f975fe8d6075600593ee253470d46d1b84801688051b09f6 \
    --hash=sha256:2aeb745f2664eb811026997c95628071137a777ea2ad296deec9cb393f0b23cf \
    --hash=sha256:2c7340497cf53490293e0c2b61011e0191633022ede0a0a964a68157a98b0fb4 \
    --hash=sha256:2ce4c93160535761f22c80b2afbc96cabc09855363a5d1a5554265b8a4c85901 \
    --hash=sha256:2d320483928c7831f0139ecb361954a26b2e2a8995681200155835dd8cd4a7d5 \
    --hash=sha256:2d637468dac23aac0e223bd52e66f8faa3b0dfcef57435460fa2107e830226cd \
    --hash=sha256:3087a430722aba8de429c2539fd2a58a9cf05238cdfefd8626460001052ca878 \
    --hash=sha256:34378db9a398b59de18cc79d942f0a907c6fc6301945e065ec888202f607aa3f \
    --hash=sha256:36bb24d4b98faacaff04af1d5e0a4285feba6ed1da6728cd34b6b6deb6bbb934 \
    --hash=sha256:38ce8f5fbd5c17dd2153d47d7c8d4108f3deda3f2b4a79b60ddc470a58faded3 \
    --hash=sha256:43d1284c1964666ee833f2badd3017dc138f53d4889043ffca66c5ce4188f188 \
    --hash=sha256:53e5854ea8003efec34adc0863c18ce91da923018354d27366f7fee7db928d7a \
    --hash=sha256:56d41d650cb8fc6cfe1d85ed7c62a0a56cbeed07bc65ca795475b914d401312a \
    --hash=sha256:5de5d80fbc0e50ff794c244e8fb7afd3eadfe0fa232ba8b162b8c551df22fcb4 \
    --hash=sha256:60f5ea17aed4262630afa43f26997ceabd6417fa05dcedf54c665f5a29193e18 \
    --hash=sha256:64967c6ddb0d4c610dfd8cb1485981b2d27972ddfb7d4bbbd9e199d2a089c450 \
    --hash=sha256:668f092bc0de8902167df6a0d5c5aedc3b4f9e43cf88eea92e9b46a2bd3968f5 \
    --hash=sha256:66fad3b7874062c2a2692f0ae6dea56d24f01b778c7f191950ca3ff997e25a88 \
    --hash=sha256:6946fe7bfb28590a1fd4061a17609c9a843952deb65dcf30d1fe725070c3e7a4 \
    --hash=sha256:71c7ca1b5f46f5dd549f56b47d47c0b709217675c23d3a7bc6aa1a69b6d9bbae \
    --hash=sha256:72299346b96b9244dabcc051b24e4653da4edfda6105544cfb10ce856a1afaac \
    --hash=sha256:7234ae9e28db64273fbbfa72caebd0a97e3bdba6b05064114741b9539ef339d0 \
    --hash=sha256:7b8ff9e0edbcee2fbf7dff0c41b9041c1901c26acf64e23adb67495012df11de \
    --hash=sha256:7cf4f996f9b1cb549bff9ea4c50813988a26ec922c95cfa85c7e4f1270447e06 \
    --hash=sha256:7f49f2834f5d006fe0f3bb10fec73b261806c50941f0cfbc08294074ffc32210 \
    --hash=sha256:83572afe48733bad7a4a9c11721d3a726c2e976d82b063fc9bdd049d76955abd \
    --hash=sha256:8526b2b7ec4db6b81efb83438be52b1264eda9a4994d867163cfe8c65581ce8d \
    --hash=sha256:8aed2bbcd6216253ef1b015763593365ee8084f621dfa53bb957c19d5e05f7cd \
    --hash=sha256:90de3477394c73481d27d2b86091c1c736053ee13ff52c42f0e151948e8578c6 \
    --hash=sha256:9261ef507f2dd74203443a472b65b5a26429eb378f975016dec7dc7305b24898 \
    --hash=sha256:9ea6c93091cbf83161a544388746a0911550bd98cb911faca3591cf5ead166ac \
    --hash=sha256:b13c8c541ce0b794add3211b3641cc0e113d707f73e06235e6fe9731bd7c45a9 \
    --hash=sha256:b18803cbdef248e7ee1be59cb277fbbe1da1faaa6f726fa5d3557904e6a3d967 \
    --hash=sha256:b878c78b2af11b879bd4f26bb0d8bda2a4c64543fdd3f28efe2c80f97f043885 \
    --hash=sha256:b8b71db96d605784e2c5ebf0788a406018ea8fdd80338491f4c83613d5cd1fec \
    --hash=sha256:b913b8e9f7ca9bec44d1eb919f591c596c61041aa357c96be55ff93169859e91 \
    --hash=sha256:c258eba62260beb33c110b03a6912cefa3635239c4ab5615b7225fb6f7b85238 \
    --hash=sha256:c47299bca4b5acaaeb32100f77b944feea151de9ef1773365a410dc3d49b945b \
    --hash=sha256:c666fefdd5613a0e99aa4516e6ff4ef87aa86cf1c7ba12a73550f4770e46b750 \
    --hash=sha256:c724e56213494c6695335577822b2d1628d102e71614de8b7eb8e30886d6a314 \
    --hash=sha256:d3b5403e82d0c7659ff1d9f956e29a3a68d094f043e9f5bc0442796fc3a4fb58 \
    --hash=sha256:d4f76868aea9cc4ce47fdbeaa904c02ee7d85dd0ad095071ae77f0bda6e62cf5 \
    --hash=sha256:d84ac0bf776b68396185bd919dd29e633d94300660335efc40b55b294b886903 \
    --hash=sha256:d8f0a8f16c4f3a5a87ca971de2631792d8cb4d570951f2000acf712f157d40db \
    --hash=sha256:dbb7b950f8c02deaffb6968994691e8589d671b7ef8396bc9d5b5c0dfbb7292f \
    --hash=sha256:dfba62cc93199ba62c376f90f2a9147d92730d301e44f88e013e50ff5edf6193 \
    --hash=sha256:e1cde50b3ec84ca6fe63ca815de183dbecb88e8adf8ada82d8ea130ef12b2b43 \
    --hash=sha256:e7ea7a08547a453fa000db96ed5714a3dc7e2b4255b9243f897921f8c10c169a \
    --hash=sha256:eef76d5796e604f9d6753fa6d323c4eb9f4e0e43f1dcca553f3e6914f1667b64 \
    --hash=sha256:f08ab7f8461c37ecfdd29ad97fb0c0780b50501bd664bb0f46b6e83ed2b9d2a7 \
    --hash=sha256:fdf4afd75c643e60ef4a96fe64fc8a9def27d2a542112332371a9e5066885f9a
    # via matplotlib
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   -r tutorials/requirements-colab.in
    #   timesfm
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
kiwisolver==1.5.1 \
    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \
    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \
    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \
    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \
    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \
    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \
    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \
    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \
    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \
    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \
    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \
    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \
    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \
    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \
    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \
    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \
    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \
    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \
    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \
    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \
    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \
    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \
    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \
    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \
    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \
    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \
    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \
    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \
    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \
    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \
    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \
    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \
    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \
    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \
    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \
    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \
    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \
    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \
    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \
    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \
    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \
    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \
    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \
    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \
    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \
    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \
    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \
    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \
    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \
    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \
    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \
    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \
    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \
    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \
    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \
    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \
    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \
    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \
    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \
    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \
    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \
    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \
    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \
    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \
    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \
    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \
    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \
    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \
    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \
    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \
    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \
    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \
    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \
    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \
    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \
    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \
    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \
    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \
    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \
    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \
    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \
    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \
    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \
    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \
    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \
    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \
    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \
    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \
    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \
    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \
    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \
    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \
    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \
    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \
    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \
    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \
    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \
    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \
    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \
    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \
    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \
    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \
    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \
    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \
    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \
    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \
    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \
    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \
    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \
    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \
    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \
    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \
    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \
    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \
    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \
    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \
    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \
    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \
    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \
    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \
    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \
    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \
    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \
    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \
    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \
    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \
    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \
    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \
    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \
    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \
    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \
    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \
    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \
    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \
    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \
    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \
    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \
    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \
    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \
    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \
    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404
    # via matplotlib
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
matplotlib==3.10.8 \
    --hash=sha256:00270d217d6b20d14b584c521f810d60c5c78406dc289859776550df837dcda7 \
    --hash=sha256:0a33deb84c15ede243aead39f77e990469fff93ad1521163305095b77b72ce4a \
    --hash=sha256:113bb52413ea508ce954a02c10ffd0d565f9c3bc7f2eddc27dfe1731e71c7b5f \
    --hash=sha256:12d90df9183093fcd479f4172ac26b322b1248b15729cb57f42f71f24c7e37a3 \
    --hash=sha256:15d30132718972c2c074cd14638c7f4592bd98719e2308bccea40e0538bc0cb5 \
    --hash=sha256:18821ace09c763ec93aef5eeff087ee493a24051936d7b9ebcad9662f66501f9 \
    --hash=sha256:1ae029229a57cd1e8fe542485f27e7ca7b23aa9e8944ddb4985d0bc444f1eca2 \
    --hash=sha256:2299372c19d56bcd35cf05a2738308758d32b9eaed2371898d8f5bd33f084aa3 \
    --hash=sha256:238b7ce5717600615c895050239ec955d91f321c209dd110db988500558e70d6 \
    --hash=sha256:24d50994d8c5816ddc35411e50a86ab05f575e2530c02752e02538122613371f \
    --hash=sha256:25d380fe8b1dc32cf8f0b1b448470a77afb195438bafdf1d858bfb876f3edf7b \
    --hash=sha256:2c1998e92cd5999e295a731bcb2911c75f597d937341f3030cc24ef2733d78a8 \
    --hash=sha256:2cf5bd12cecf46908f286d7838b2abc6c91cda506c0445b8223a7c19a00df008 \
    --hash=sha256:32f8dce744be5569bebe789e46727946041199030db8aeb2954d26013a0eb26b \
    --hash=sha256:37b3c1cc42aa184b3f738cfa18c1c1d72fd496d85467a6cf7b807936d39aa656 \
    --hash=sha256:3a48a78d2786784cc2413e57397981fb45c79e968d99656706018d6e62e57958 \
    --hash=sha256:3ab4aabc72de4ff77b3ec33a6d78a68227bf1123465887f9905ba79184a1cc04 \
    --hash=sha256:3c624e43ed56313651bc18a47f838b60d7b8032ed348911c54906b130b20071b \
    --hash=sha256:3f2e409836d7f5ac2f1c013110a4d50b9f7edc26328c108915f9075d7d7a91b6 \
    --hash=sha256:3f5c3e4da343bba819f0234186b9004faba952cc420fbc522dc4e103c1985908 \
    --hash=sha256:41703cc95688f2516b480f7f339d8851a6035f18e100ee6a32bc0b8536a12a9c \
    --hash=sha256:495672de149445ec1b772ff2c9ede9b769e3cb4f0d0aa7fa730d7f59e2d4e1c1 \
    --hash=sha256:4cf267add95b1c88300d96ca837833d4112756045364f5c734a2276038dae27d \
    --hash=sha256:56271f3dac49a88d7fca5060f004d9d22b865f743a12a23b1e937a0be4818ee1 \
    --hash=sha256:595ba4d8fe983b88f0eec8c26a241e16d6376fe1979086232f481f8f3f67494c \
    --hash=sha256:5f62550b9a30afde8c1c3ae450e5eb547d579dd69b25c2fc7a1c67f934c1717a \
    --hash=sha256:646d95230efb9ca614a7a594d4fcacde0ac61d25e37dd51710b36477594963ce \
    --hash=sha256:64fcc24778ca0404ce0cb7b6b77ae1f4c7231cdd60e6778f999ee05cbd581b9a \
    --hash=sha256:6be43b667360fef5c754dda5d25a32e6307a03c204f3c0fc5468b78fa87b4160 \
    --hash=sha256:6da7c2ce169267d0d066adcf63758f0604aa6c3eebf67458930f9d9b79ad1db1 \
    --hash=sha256:83d282364ea9f3e52363da262ce32a09dfe241e4080dcedda3c0db059d3c1f11 \
    --hash=sha256:9153c3292705be9f9c64498a8872118540c3f4123d1a1c840172edf262c8be4a \
    --hash=sha256:99eefd13c0dc3b3c1b4d561c1169e65fe47aab7b8158754d7c084088e2329466 \
    --hash=sha256:a0a7f52498f72f13d4a25ea70f35f4cb60642b466cbb0a9be951b5bc3f45a486 \
    --hash=sha256:a2b336e2d91a3d7006864e0990c83b216fcdca64b5a6484912902cef87313d78 \
    --hash=sha256:a48f2b74020919552ea25d222d5cc6af9ca3f4eb43a93e14d068457f545c2a17 \
    --hash=sha256:ad3d9833a64cf48cc4300f2b406c3d0f4f4724a91c0bd5640678a6ba7c102077 \
    --hash=sha256:b44d07310e404ba95f8c25aa5536f154c0a8ec473303535949e52eb71d0a1565 \
    --hash=sha256:b53285e65d4fa4c86399979e956235deb900be5baa7fc1218ea67fbfaeaadd6f \
    --hash=sha256:b5a2b97dbdc7d4f353ebf343744f1d1f1cca8aa8bfddb4262fcf4306c3761d50 \
    --hash=sha256:b9a5ca4ac220a0cdd1ba6bcba3608547117d30468fefce49bb26f55c1a3d5c58 \
    --hash=sha256:bab485bcf8b1c7d2060b4fcb6fc368a9e6f4cd754c9c2fea281f4be21df394a2 \
    --hash=sha256:c108a1d6fa78a50646029cb6d49808ff0fc1330fda87fa6f6250c6b5369b6645 \
    --hash=sha256:d56a1efd5bfd61486c8bc968fa18734464556f0fb8e51690f4ac25d85cbbbbc2 \
    --hash=sha256:d9050fee89a89ed57b4fb2c1bfac9a3d0c57a0d55aed95949eedbc42070fea39 \
    --hash=sha256:dd80ecb295460a5d9d260df63c43f4afbdd832d725a531f008dad1664f458adf \
    --hash=sha256:e8ea3e2d4066083e264e75c829078f9e149fa119d27e19acd503de65e0b13149 \
    --hash=sha256:eb3823f11823deade26ce3b9f40dcb4a213da7a670013929f31d5f5ed1055b22 \
    --hash=sha256:ee40c27c795bda6a5292e9cff9890189d32f7e3a0bf04e0e3c9430c4a00c37df \
    --hash=sha256:efb30e3baaea72ce5928e32bab719ab4770099079d66726a62b11b1ef7273be4 \
    --hash=sha256:f254d118d14a7f99d616271d6c3c27922c092dac11112670b157798b89bf4933 \
    --hash=sha256:f89c151aab2e2e23cb3fe0acad1e8b82841fd265379c4cecd0f3fcb34c15e0f6 \
    --hash=sha256:f97aeb209c3d2511443f8797e3e5a569aebb040d4f8bc79aa3ee78a8fb9e3dd8 \
    --hash=sha256:f9b587c9c7274c1613a30afabf65a272114cd6cdbe67b3406f818c79d7ab2e2a \
    --hash=sha256:fb061f596dad3a0f52b60dc6a5dec4a0c300dec41e058a7efe09256188d170b7
    # via -r tutorials/requirements-colab.in
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   -r tutorials/requirements-colab.in
    #   contourpy
    #   matplotlib
    #   pandas
    #   timesfm
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   matplotlib
pandas==3.0.5 \
    --hash=sha256:08d24fe11a17dc33bd6e937dc9c665f9cba08fbdc9f657f405713515febe300d \
    --hash=sha256:0d298e951f23016ce4699951d044ae6418dbc91bf68cefca0f77666fcbb4e5c6 \
    --hash=sha256:0fac0010c75e4efb6b99e249c183a8993ce0dc95c240f9b120a5e67c727b7928 \
    --hash=sha256:1c10461f6eeb35d8f05b6184c65c8b9991663b66c46b1d559b682cb34ae7c6ea \
    --hash=sha256:25ff585b972a18ef1fe9ffa3ac6544d9950508aa76832e5147640b6022821e49 \
    --hash=sha256:2946e77e4a53cd248cbde631a12f0e51c8324ce354c3eba4d20147c1ad6f4282 \
    --hash=sha256:2a29c53d85ea98c5e792c59ef82ee9fbe6ca902c0d0adb6b23f45ef894cd7bf6 \
    --hash=sha256:2c0cf1dd9b55a22d105fc46c1b489af3bd42264fcba7c66297bf47a9a1d9c78a \
    --hash=sha256:2f264fc46911cc8131a7322a16199bbf8e353d27c10bb211f5bd0c814324dc36 \
    --hash=sha256:303da736987d481074ca720ada325f8bd80c64ebc2d45ed79b29df3aaa4a26ca \
    --hash=sha256:3b2801bbb049d0136f6c213eae02b5fca969384fc2064dd728d8620552aa49da \
    --hash=sha256:3c5015fd1730fbf883647e88068176c839c102cea883ba1769a6f4593bfc1f8c \
    --hash=sha256:3c5ed2e7c06e91d340dfd091d7934f9bc82e4a36b95f647f090b9d1c9ac649da \
    --hash=sha256:4b11c36e218331d0387cbe3a0a5f75162357a1d92d57b2b08a336ff94b19b2be \
    --hash=sha256:5183427f5a8156d480f30333777bc978be93650a49a7c01db26adffe95b31e85 \
    --hash=sha256:53730687fcd161883b24e10411c06d6a4c0f2275d2faf3bb2bc25deb4ba8007c \
    --hash=sha256:66266d3442a5e8b3c90274c2b8b230bee42dd1c286bc822cc2f9f2c7e12b883e \
    --hash=sha256:679f4e85b30ddb1515458ab1e788d3e260eae369b1f78da7a3aa4cac8ebf4a2a \
    --hash=sha256:71ecc8fb7ed1a7aa4392316b5309a6347e8e7f832f38fd897846b3a1457a9298 \
    --hash=sha256:73fa87b08a7ef706f8aafda39ddaccf2a99047bea62d8c88a0361bcafb2237bc \
    --hash=sha256:80a611068e8a3ac23f7398c6c14eb46dc974e5cc9997f653e2dcfd1da74edd41 \
    --hash=sha256:960d3ebcf249f75206899fcd2c6de53f736b7265759ced0d3e559df0b8b709b0 \
    --hash=sha256:9e94c2c5ca43bd3ca32bf64d32308887b65e5f9bfd8023ea52755107a999f93b \
    --hash=sha256:a5ad3b02ed6bc7d7ae9b70804b2c6aa31827489d150f8e623ce82491b82085d7 \
    --hash=sha256:b1261758dfb6cf12c3cff8300e21cefad30e7ec709abb4c24ac7318e6a52462a \
    --hash=sha256:b173f5951ff6b8b0ec7675e20dff3c97b7e7a57dfcce387c2d7c5afe87cb7899 \
    --hash=sha256:b2acb4650527eec6822c3dadb2b771277b65e7dae7a267d4bccf65fd1bb3fbce \
    --hash=sha256:b58b1b39d46a5862e3fb18f50d1a201398619d16a0f9f73f57eea5583cf0e63c \
    --hash=sha256:b86765f268b56f7e665b93bce9d5df69dee7f99e595cf8fb839483ab315942a3 \
    --hash=sha256:c1c05a767fe8e5b4fe9e1c29806829c582052eaedb9120a3da83ba3f69e24a5b \
    --hash=sha256:c2e26bb46934b8a2ca0c3de1d3d606fc5f6746584791b2db264d58cf370e08dc \
    --hash=sha256:c597ecf5616b5c420372c1d4d4c00dbbfba7398bea857dcc984347e1ea48417b \
    --hash=sha256:cce3a9d11d2b1f82c69a27ec1f4948a170e2c403c4bbfa8cca62e3fdebe2ef3a \
    --hash=sha256:cd8f7c6dc98527058ee6264219343f5392240a6f1bfa654fc5d79023020d0c92 \
    --hash=sha256:cf52e1f61d229496da17dc7ab54acdee627357e7008fd4fecba3d0ba2937fa58 \
    --hash=sha256:d373ce03ffd84010ed9839fa73672a9c8256990532e158440c0085db7d914b34 \
    --hash=sha256:db172144bb56422bd157812f3b021eacc255451470b31e2c633c349490a1cfee \
    --hash=sha256:dca3734d6ab7c906e6730f0788b0a1dbb9f2467731f9711f77995c8e9d62d712 \
    --hash=sha256:e2759e890db96dfcffdbd9b86c3c2cb6afaf58def482820317e06163ec1066cd \
    --hash=sha256:e819dd5f62966b481a8cb649d3299ebd886a1ea91ed5a99bf7ce77c98d18ab94 \
    --hash=sha256:ef01af4d8dc6cd2c8d6c7736f149574ef93fe043811eeb5e445f2647154b5040 \
    --hash=sha256:fa290c16964d4963fbfbc358928239cf3bd755b20e988ce944877def2f44471d
    # via -r tutorials/requirements-colab.in
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via matplotlib
pyparsing==3.3.3 \
    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \
    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4
    # via matplotlib
python-dateutil==2.9.0.post0 \
    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \
    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427
    # via
    #   matplotlib
    #   pandas
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via huggingface-hub
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via huggingface-hub
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   -r tutorials/requirements-colab.in
    #   timesfm
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
six==1.17.0 \
    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \
    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81
    # via python-dateutil
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
timesfm==3.0.2 \
    --hash=sha256:a602deba10b9323132d0bc694c08e6cfaf144d7b9e4554da33155b6a9cb0ea5c \
    --hash=sha256:cfa3d22ddc3f019f0d9d95f19055428c4a4713526dafd7e5d1a7769782e7613f
    # via -r tutorials/requirements-colab.in
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via -r tutorials/requirements-colab.in
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via huggingface-hub
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    try:
        import google
    except ImportError:
        google = types.ModuleType("google")
        google.__path__ = []
        sys.modules["google"] = google
    _colab = types.ModuleType("google.colab")
    _files = types.ModuleType("google.colab.files")
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `numpy`, `pandas` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'timesfm-forecasting-pipeline',
    'repository_revision': 'fc4c39fe98189aebe373c03ca58313e6b4e02d03',
    'embedded_module': 'src/timesfm_forecasting/model.py',
    'embedded_modules': ['src/timesfm_forecasting/config.py', 'src/timesfm_forecasting/errors.py', 'src/timesfm_forecasting/data.py', 'src/timesfm_forecasting/model.py', 'src/timesfm_forecasting/forecasting.py', 'src/timesfm_forecasting/evaluation.py', 'src/timesfm_forecasting/provenance.py'],
    'module_sha256': '5044003eac6a27708168dd7caebd771f90f2b7ad279723368840352a4f29626b',
    'generator': 'build_notebook.py/2.2',
    'notebook_spec': '2.0',
}
import torch, numpy, pandas
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/timesfm_forecasting/` @ `fc4c39fe9818`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 7 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/7:** `src/timesfm_forecasting/config.py`

In [ ]:
"""Forecast request configuration.

The configuration is a frozen dataclass so that a request can be hashed, serialised into
provenance and compared across the sample and BYOD paths. Every ceiling is named here once and
re-used by validation, the model wrapper and the notebook's ceiling print.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any, Literal

__all__ = [
    "ForecastConfig",
    "TRAINED_QUANTILES",
    "MEDIAN_SLOT",
    "MAX_HORIZON",
    "MAX_CONTEXT_POINTS",
    "MIN_CONTEXT_POINTS",
    "DEFAULT_QUANTILE_LEVELS",
]

#: The nine deciles TimesFM 3.0 was trained to emit (``config.json`` ``quantiles``). The model's
#: quantile array has exactly these nine slots, in this order; slot ``MEDIAN_SLOT`` (index 4) is the
#: median and is the point forecast. There is no separate mean head. Any other level is refused
#: rather than interpolated.
TRAINED_QUANTILES: tuple[float, ...] = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
MEDIAN_SLOT = TRAINED_QUANTILES.index(0.5)
DEFAULT_QUANTILE_LEVELS: tuple[float, ...] = (0.1, 0.5, 0.9)

#: Longest forecast one request may ask for. TimesFM 3.0 decodes the horizon autoregressively in
#: 64-point output patches with no fixed upper bound; the pipeline keeps a ceiling so a tutorial
#: request stays cheap and a horizon far beyond the context's cycle is never silently accepted.
MAX_HORIZON = 512
#: TimesFM 3.0 context limit (``timesfm3`` ``_MAX_CONTEXT_LENGTH = 15360``; the forecaster truncates
#: longer contexts to their most recent points). Longer histories are truncated here first and the
#: truncation is reported, never silent.
MAX_CONTEXT_POINTS = 15_360
#: Fewest history points the pipeline will pass to the model per series. The model pads shorter
#: inputs with zeros; below this many real points the forecast is dominated by padding.
MIN_CONTEXT_POINTS = 16

NanPolicy = Literal["refuse", "interpolate"]
GapPolicy = Literal["refuse", "fill"]


def _is_trained_quantile(level: float) -> bool:
    return any(abs(level - q) < 1e-9 for q in TRAINED_QUANTILES)


@dataclass(frozen=True)
class ForecastConfig:
    """One forecasting request.

    Attributes
    ----------
    horizon
        Number of future steps to forecast (1..``MAX_HORIZON``).
    context_length
        Most recent history points passed to the model per series
        (``MIN_CONTEXT_POINTS``..``MAX_CONTEXT_POINTS``). Shorter series are passed whole.
    quantile_levels
        Quantiles to export, each one of ``TRAINED_QUANTILES``; ``0.5`` is always carried
        because the point forecast *is* the median.
    timestamp_column, value_column, series_id_column
        Column names of the long-format input. ``series_id_column=None`` means the file holds
        one series, labelled ``"series"``.
    season_length
        Period in steps for the seasonal-naive baseline and MASE scaling; ``None`` disables the
        seasonal baseline and scales MASE by the one-step naive error.
    nan_policy
        ``"refuse"`` rejects any missing value; ``"interpolate"`` fills interior gaps linearly
        and drops leading missing values, reporting every change.
    gap_policy
        ``"refuse"`` rejects a series whose timestamps skip grid points; ``"fill"`` inserts the
        missing timestamps (then ``nan_policy`` decides what happens to their values).
    """

    horizon: int = 24
    context_length: int = 512
    quantile_levels: tuple[float, ...] = DEFAULT_QUANTILE_LEVELS
    timestamp_column: str = "timestamp"
    value_column: str = "value"
    series_id_column: str | None = None
    season_length: int | None = None
    nan_policy: NanPolicy = "refuse"
    gap_policy: GapPolicy = "refuse"

    def __post_init__(self) -> None:
        if isinstance(self.horizon, bool) or not isinstance(self.horizon, int):
            raise ValueError(f"horizon must be an int, got {self.horizon!r}")
        if not 1 <= self.horizon <= MAX_HORIZON:
            raise ValueError(f"horizon must be in 1..{MAX_HORIZON}, got {self.horizon}")
        if isinstance(self.context_length, bool) or not isinstance(self.context_length, int):
            raise ValueError(f"context_length must be an int, got {self.context_length!r}")
        if not MIN_CONTEXT_POINTS <= self.context_length <= MAX_CONTEXT_POINTS:
            raise ValueError(
                f"context_length must be in {MIN_CONTEXT_POINTS}..{MAX_CONTEXT_POINTS}, "
                f"got {self.context_length}"
            )
        levels = tuple(float(q) for q in self.quantile_levels)
        bad = [q for q in levels if not _is_trained_quantile(q)]
        if bad:
            raise ValueError(
                f"quantile_levels {bad} are not in the trained grid {TRAINED_QUANTILES}; "
                "the pipeline does not interpolate quantiles"
            )
        if 0.5 not in levels:
            levels = tuple(sorted({*levels, 0.5}))
        object.__setattr__(self, "quantile_levels", tuple(sorted(set(levels))))
        if self.season_length is not None and (
            isinstance(self.season_length, bool)
            or not isinstance(self.season_length, int)
            or self.season_length < 1
        ):
            raise ValueError(f"season_length must be a positive int or None, got {self.season_length!r}")
        if self.nan_policy not in ("refuse", "interpolate"):
            raise ValueError(f"nan_policy must be 'refuse' or 'interpolate', got {self.nan_policy!r}")
        if self.gap_policy not in ("refuse", "fill"):
            raise ValueError(f"gap_policy must be 'refuse' or 'fill', got {self.gap_policy!r}")
        for name in ("timestamp_column", "value_column"):
            if not isinstance(getattr(self, name), str) or not getattr(self, name):
                raise ValueError(f"{name} must be a non-empty string")
        names = [self.timestamp_column, self.value_column] + (
            [self.series_id_column] if self.series_id_column else []
        )
        if len(set(names)) != len(names):
            raise ValueError(f"column names must be distinct, got {names}")

    def to_dict(self) -> dict[str, Any]:
        out = asdict(self)
        out["quantile_levels"] = list(self.quantile_levels)
        return out

**Module 2/7:** `src/timesfm_forecasting/errors.py` (carried verbatim; see the note above)

In [ ]:
"""Exception types with stable machine-readable codes.

Every refusal raised by the pipeline names the *rule* that refused the input and, for data
errors, the *source* (file path or table name) it was refusing, so a learner or an executor can
read ``[RULE] source: reason`` and act on it without a traceback.
"""

from __future__ import annotations

from typing import Any


class TimesFMPipelineError(Exception):
    """Base class for every error this package raises deliberately."""


class ValidationError(TimesFMPipelineError, ValueError):
    """An input was refused by a named validation rule.

    Parameters
    ----------
    code
        The rule identifier, e.g. ``MIN_HISTORY``.
    source
        The file path or table name the rule was applied to.
    message
        A plain-language reason, naming the offending series/row/column where known.
    details
        Machine-readable context (counts, names, limits) for executors and tests.
    """

    def __init__(
        self, code: str, source: str, message: str, details: dict[str, Any] | None = None
    ) -> None:
        self.code = code
        self.source = source
        self.message = message
        self.details = dict(details or {})
        super().__init__(f"[{code}] {source}: {message}")


class ModelSourceError(TimesFMPipelineError, ValueError):
    """A model identifier or revision other than the pinned one was requested."""


class ModelIntegrityError(TimesFMPipelineError, RuntimeError):
    """A staged snapshot does not match the committed manifest."""


class HubUnavailableError(TimesFMPipelineError, RuntimeError):
    """The Hugging Face Hub could not be reached to stage a missing file."""

**Module 3/7:** `src/timesfm_forecasting/data.py` (carried verbatim; see the note above)

In [ ]:
"""Loading and validation of long-format time-series tables (sample path and BYOD alike).

The contract is deliberately small: a CSV with a timestamp column and a numeric value column,
optionally a series-id column. Validation establishes the sampling frequency, finds gaps and
missing values, applies the configured policies while *reporting* every change, enforces the
length ceilings, and returns a normalised frame plus an input manifest. Every refusal is a
:class:`ValidationError` whose text reads ``[RULE] source: reason``.
"""

from __future__ import annotations

import csv
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import MAX_CONTEXT_POINTS, MIN_CONTEXT_POINTS, ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import ValidationError` removed — names are kernel globals defined by the carried modules

__all__ = [
    "MIN_OBSERVATIONS",
    "MAX_SERIES",
    "MAX_ROWS",
    "MAX_NAN_FRACTION",
    "INPUT_SCHEMA",
    "RULES",
    "FrequencyInfo",
    "SeriesReport",
    "ValidationResult",
    "infer_frequency",
    "future_timestamps",
    "load_series_csv",
    "validate_series",
    "min_history_required",
]

#: Normalised column names of every frame the package passes around after validation.
ID_COLUMN = "series_id"
TIMESTAMP_COLUMN = "timestamp"
VALUE_COLUMN = "value"
SINGLE_SERIES_ID = "series"

#: Fewest rows for which a sampling interval can be inferred and checked.
MIN_OBSERVATIONS = 3
MAX_SERIES = 1_000
MAX_ROWS = 2_000_000
#: Largest share of a series that ``nan_policy="interpolate"`` may fill.
MAX_NAN_FRACTION = 0.2

INPUT_SCHEMA = {
    "format": "CSV, long format (one row per observation)",
    "timestamp_column": "ISO-8601 or any pandas-parseable timestamp; one per row",
    "value_column": "finite numeric target",
    "series_id_column": "optional; distinct values become independent series",
    "frequency": "inferred: one fixed interval, or a calendar interval (month/quarter/year)",
    "min_rows_per_series": f"horizon + {MIN_CONTEXT_POINTS} (see min_history_required)",
    "max_series": MAX_SERIES,
    "max_rows": MAX_ROWS,
    "max_context_points": MAX_CONTEXT_POINTS,
}

#: Rule identifiers and what each refuses. Kept in one place so the notebook can print them.
RULES = {
    "FILE_NOT_FOUND": "the CSV path does not exist or is not a file",
    "FILE_EMPTY": "the CSV has no data rows",
    "DUPLICATE_HEADER": "two header cells share a name; the table is ambiguous",
    "REQUIRED_COLUMNS": "a configured column is missing from the header",
    "SERIES_ID_EMPTY": "a series id is blank or null",
    "TIMESTAMP_PARSE": "a timestamp cannot be parsed",
    "TIMESTAMP_DUPLICATE": "one series has two rows for the same timestamp",
    "VALUE_NUMERIC": "a value is not numeric",
    "VALUE_INFINITE": "a value is +/-inf",
    "VALUE_MISSING": "a value is missing and nan_policy is 'refuse', or cannot be interpolated",
    "VALUE_MISSING_FRACTION": f"more than {MAX_NAN_FRACTION:.0%} of a series would be interpolated",
    "MIN_OBSERVATIONS": f"a series has fewer than {MIN_OBSERVATIONS} rows",
    "FREQUENCY_IRREGULAR": "timestamps are neither one fixed interval nor a calendar interval",
    "FREQUENCY_GAP": "a series skips grid points and gap_policy is 'refuse'",
    "FREQUENCY_MIXED": "series in one file have different sampling intervals",
    "MIN_HISTORY": "a series is shorter than horizon + the minimum context",
    "MAX_SERIES": f"more than {MAX_SERIES} series",
    "MAX_ROWS": f"more than {MAX_ROWS} rows",
}

_CALENDAR_BASES = {
    "MS", "ME", "BMS", "BME", "SMS", "SME", "QS", "QE", "BQS", "BQE", "YS", "YE", "BYS", "BYE",
    "B", "C",
}


def _fail(code: str, source: str, message: str, **details: Any) -> None:
    raise ValidationError(code, source, message, details)


def min_history_required(config: ForecastConfig) -> int:
    """Rows a series needs so that a holdout of ``horizon`` leaves a usable context."""
    return config.horizon + MIN_CONTEXT_POINTS


@dataclass(frozen=True)
class FrequencyInfo:
    """How timestamps are spaced.

    ``kind`` is ``"fixed"`` (one constant interval, ``step`` set), ``"calendar"`` (a pandas
    calendar alias such as ``MS``, ``step`` None), ``"gapped"`` (fixed grid with skipped points)
    or ``"irregular"``.
    """

    kind: str
    alias: str | None
    step: pd.Timedelta | None
    n_gaps: int = 0
    first_gap_after: pd.Timestamp | None = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "kind": self.kind,
            "alias": self.alias,
            "step_seconds": None if self.step is None else float(self.step.total_seconds()),
            "n_gaps": int(self.n_gaps),
            "first_gap_after": None if self.first_gap_after is None else str(self.first_gap_after),
        }


def _calendar_alias(stamps: pd.DatetimeIndex) -> str | None:
    if len(stamps) < 3:
        return None
    try:
        alias = pd.infer_freq(stamps)
    except (TypeError, ValueError):
        return None
    if not alias:
        return None
    base = alias.split("-", 1)[0].lstrip("0123456789")
    return alias if base in _CALENDAR_BASES else None


def _step_alias(step: pd.Timedelta) -> str:
    """A readable alias for a fixed step: whole days as ``<n>D``, otherwise pandas' own spelling."""
    day = pd.Timedelta(days=1)
    if step >= day and step % day == pd.Timedelta(0):
        return f"{step // day}D"
    return pd.tseries.frequencies.to_offset(step).freqstr


def infer_frequency(stamps: pd.DatetimeIndex) -> FrequencyInfo:
    """Classify the spacing of a sorted, duplicate-free timestamp index."""
    if len(stamps) < 2:
        return FrequencyInfo("irregular", None, None)
    # pandas >= 2 stores an index in its parsed unit (s, ms, us or ns); compare in nanoseconds.
    diffs = np.diff(stamps.as_unit("ns").asi8)
    unique = np.unique(diffs)
    if len(unique) == 1:
        step = pd.Timedelta(int(unique[0]), unit="ns")
        return FrequencyInfo("fixed", _step_alias(step), step)
    alias = _calendar_alias(stamps)
    if alias is not None:
        return FrequencyInfo("calendar", alias, None)
    step_ns = int(unique[0])
    if step_ns > 0 and np.all(diffs % step_ns == 0):
        multiples = diffs // step_ns
        n_gaps = int(np.sum(multiples - 1))
        first = int(np.argmax(multiples > 1))
        step = pd.Timedelta(step_ns, unit="ns")
        return FrequencyInfo("gapped", _step_alias(step), step, n_gaps, stamps[first])
    return FrequencyInfo("irregular", None, None)


def future_timestamps(last: pd.Timestamp, frequency: FrequencyInfo, horizon: int) -> pd.DatetimeIndex:
    """The ``horizon`` grid points after ``last`` for a fixed or calendar frequency."""
    if frequency.kind == "fixed" and frequency.step is not None:
        return pd.DatetimeIndex([last + frequency.step * (k + 1) for k in range(horizon)])
    if frequency.alias is None:
        raise ValueError("future timestamps need a fixed or calendar frequency")
    return pd.date_range(start=last, periods=horizon + 1, freq=frequency.alias)[1:]


@dataclass(frozen=True)
class SeriesReport:
    """What validation observed and changed for one series."""

    series_id: str
    n_rows_in: int
    n_rows_out: int
    start: str
    end: str
    n_leading_missing_dropped: int = 0
    n_interpolated: int = 0
    n_gap_points_filled: int = 0
    truncated_from: int | None = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "series_id": self.series_id,
            "n_rows_in": self.n_rows_in,
            "n_rows_out": self.n_rows_out,
            "start": self.start,
            "end": self.end,
            "n_leading_missing_dropped": self.n_leading_missing_dropped,
            "n_interpolated": self.n_interpolated,
            "n_gap_points_filled": self.n_gap_points_filled,
            "truncated_from": self.truncated_from,
        }


@dataclass(frozen=True)
class ValidationResult:
    """The normalised table plus the evidence that it passed every rule."""

    frame: pd.DataFrame
    frequency: FrequencyInfo
    series: dict[str, SeriesReport]
    source: str
    config: ForecastConfig
    changes: list[str] = field(default_factory=list)

    @property
    def series_ids(self) -> list[str]:
        return list(self.series)

    def with_frame(self, frame: pd.DataFrame) -> ValidationResult:
        """The same validated contract over a sub-table (e.g. the history half of a holdout)."""
        return replace(self, frame=frame)

    def manifest(self) -> dict[str, Any]:
        """Machine-readable input manifest (what was validated, under which rules and ceilings)."""
        return {
            "source": self.source,
            "schema": INPUT_SCHEMA,
            "columns": {
                "timestamp": self.config.timestamp_column,
                "value": self.config.value_column,
                "series_id": self.config.series_id_column,
            },
            "frequency": self.frequency.to_dict(),
            "n_series": len(self.series),
            "n_rows": int(len(self.frame)),
            "series": [report.to_dict() for report in self.series.values()],
            "ceilings": {
                "min_observations": MIN_OBSERVATIONS,
                "min_history_required": min_history_required(self.config),
                "max_series": MAX_SERIES,
                "max_rows": MAX_ROWS,
                "max_context_points": MAX_CONTEXT_POINTS,
                "context_length": self.config.context_length,
                "horizon": self.config.horizon,
            },
            "policies": {"nan_policy": self.config.nan_policy, "gap_policy": self.config.gap_policy},
            "changes": list(self.changes),
            "verdict": "accepted",
        }


# ---------------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------------


def _header_cells(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        try:
            return next(reader)
        except StopIteration:
            return []


def load_series_csv(path: str | Path, config: ForecastConfig) -> pd.DataFrame:
    """Read a CSV and check its header before any parsing of values.

    Returns the raw frame restricted to the configured columns; :func:`validate_series` does the
    rest. Refuses a missing file, an empty file, duplicate header names and missing columns,
    naming the file in every case.
    """
    path = Path(path)
    source = str(path)
    if not path.is_file():
        _fail("FILE_NOT_FOUND", source, "no such file.", path=source)
    header = _header_cells(path)
    if not header:
        _fail("FILE_EMPTY", source, "the file has no header row.")
    names = [cell.strip() for cell in header]
    seen: dict[str, int] = {}
    for name in names:
        seen[name] = seen.get(name, 0) + 1
    duplicates = sorted(name for name, count in seen.items() if count > 1)
    if duplicates:
        _fail(
            "DUPLICATE_HEADER",
            source,
            f"header names {duplicates} appear more than once; rename them so each column is unambiguous.",
            duplicates=duplicates,
        )
    required = [config.timestamp_column, config.value_column] + (
        [config.series_id_column] if config.series_id_column else []
    )
    missing = [name for name in required if name not in names]
    if missing:
        _fail(
            "REQUIRED_COLUMNS",
            source,
            f"required column(s) {missing} not in header {names}; set the column fields to match your file.",
            missing=missing,
            present=names,
        )
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    frame.columns = [str(c).strip() for c in frame.columns]
    frame = frame[required]
    if len(frame) == 0:
        _fail("FILE_EMPTY", source, "the file has a header but no data rows.")
    return frame


# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------


def _parse_values(raw: pd.Series, source: str, series_id: str) -> np.ndarray:
    # pandas >= 3 keeps a float NaN as a missing string under astype(str); normalise explicitly.
    text = pd.Series(["" if pd.isna(x) else str(x).strip() for x in raw], index=raw.index, dtype=object)
    blank = text.isin(["", "nan", "NaN", "NA", "null", "None"])
    values = pd.to_numeric(text.where(~blank, None), errors="coerce")
    unparsable = values.isna() & ~blank
    if unparsable.any():
        bad = text[unparsable].iloc[0]
        _fail(
            "VALUE_NUMERIC",
            source,
            f"series {series_id!r} has a non-numeric value {bad!r} at row {int(unparsable.idxmax())}.",
            series_id=series_id,
            example=bad,
        )
    out = values.to_numpy(dtype=float)
    if np.isinf(out).any():
        _fail(
            "VALUE_INFINITE",
            source,
            f"series {series_id!r} contains an infinite value at row {int(np.argmax(np.isinf(out)))}.",
            series_id=series_id,
        )
    return out


def _apply_missing_policy(
    values: np.ndarray, config: ForecastConfig, source: str, series_id: str
) -> tuple[np.ndarray, int, int]:
    """Return (values, n_leading_dropped, n_interpolated) under ``config.nan_policy``."""
    missing = np.isnan(values)
    if not missing.any():
        return values, 0, 0
    if config.nan_policy == "refuse":
        first = int(np.argmax(missing))
        _fail(
            "VALUE_MISSING",
            source,
            f"series {series_id!r} has {int(missing.sum())} missing value(s), first at position {first}; "
            "remove them or set nan_policy='interpolate'.",
            series_id=series_id,
            n_missing=int(missing.sum()),
            first_missing=first,
        )
    first_valid = int(np.argmax(~missing))
    if missing.all():
        _fail("VALUE_MISSING", source, f"series {series_id!r} has no non-missing value.", series_id=series_id)
    values = values[first_valid:]
    missing = missing[first_valid:]
    if missing[-1]:
        _fail(
            "VALUE_MISSING",
            source,
            f"series {series_id!r} ends with a missing value; interpolation needs a later observation.",
            series_id=series_id,
        )
    n_interp = int(missing.sum())
    if n_interp / len(values) > MAX_NAN_FRACTION:
        _fail(
            "VALUE_MISSING_FRACTION",
            source,
            f"series {series_id!r} would have {n_interp}/{len(values)} values interpolated, above the "
            f"{MAX_NAN_FRACTION:.0%} ceiling.",
            series_id=series_id,
            n_missing=n_interp,
            n_rows=len(values),
        )
    if n_interp:
        idx = np.arange(len(values))
        values = values.copy()
        values[missing] = np.interp(idx[missing], idx[~missing], values[~missing])
    return values, first_valid, n_interp


def validate_series(
    frame: pd.DataFrame, config: ForecastConfig, *, source: str = "<frame>"
) -> ValidationResult:
    """Apply every rule in :data:`RULES` and return the normalised table.

    The returned frame has columns ``series_id``, ``timestamp`` (datetime64), ``value`` (float),
    sorted by series then time. Nothing here touches the model.
    """
    required = [config.timestamp_column, config.value_column] + (
        [config.series_id_column] if config.series_id_column else []
    )
    missing_cols = [c for c in required if c not in frame.columns]
    if missing_cols:
        _fail(
            "REQUIRED_COLUMNS",
            source,
            f"required column(s) {missing_cols} not in {list(frame.columns)}.",
            missing=missing_cols,
        )
    if len(frame) == 0:
        _fail("FILE_EMPTY", source, "the table has no rows.")
    if len(frame) > MAX_ROWS:
        _fail("MAX_ROWS", source, f"{len(frame)} rows exceed the {MAX_ROWS}-row ceiling.", n_rows=len(frame))

    if config.series_id_column:
        ids = frame[config.series_id_column].astype(str).str.strip()
        blank = ids.isin(["", "nan", "None"]) | frame[config.series_id_column].isna()
        if blank.any():
            _fail(
                "SERIES_ID_EMPTY",
                source,
                f"{int(blank.sum())} row(s) have a blank series id (first at row {int(blank.idxmax())}).",
                n_blank=int(blank.sum()),
            )
    else:
        ids = pd.Series([SINGLE_SERIES_ID] * len(frame), index=frame.index)
    series_ids = list(dict.fromkeys(ids.tolist()))
    if len(series_ids) > MAX_SERIES:
        _fail(
            "MAX_SERIES", source, f"{len(series_ids)} series exceed the {MAX_SERIES}-series ceiling.",
            n_series=len(series_ids),
        )

    try:
        stamps = pd.to_datetime(frame[config.timestamp_column], errors="raise")
    except (ValueError, TypeError, pd.errors.ParserError) as exc:
        _fail("TIMESTAMP_PARSE", source, f"column {config.timestamp_column!r} could not be parsed: {exc}")
    if stamps.isna().any():
        row = int(stamps.isna().idxmax())
        _fail("TIMESTAMP_PARSE", source, f"column {config.timestamp_column!r} is empty at row {row}.", row=row)
    if getattr(stamps.dt, "tz", None) is not None:
        stamps = stamps.dt.tz_convert("UTC").dt.tz_localize(None)

    changes: list[str] = []
    reports: dict[str, SeriesReport] = {}
    parts: list[pd.DataFrame] = []
    frequencies: dict[str, FrequencyInfo] = {}
    needed = min_history_required(config)
    for series_id in series_ids:
        mask = (ids == series_id).to_numpy()
        part = pd.DataFrame(
            {TIMESTAMP_COLUMN: stamps[mask].to_numpy(), "_raw": frame.loc[mask, config.value_column].to_numpy()}
        )
        n_in = len(part)
        part = part.sort_values(TIMESTAMP_COLUMN, kind="mergesort").reset_index(drop=True)
        dup = part[TIMESTAMP_COLUMN].duplicated()
        if dup.any():
            when = part.loc[dup.idxmax(), TIMESTAMP_COLUMN]
            _fail(
                "TIMESTAMP_DUPLICATE",
                source,
                f"series {series_id!r} has {int(dup.sum())} duplicate timestamp(s), first at {when}.",
                series_id=series_id,
                n_duplicates=int(dup.sum()),
            )
        if n_in < MIN_OBSERVATIONS:
            _fail(
                "MIN_OBSERVATIONS",
                source,
                f"series {series_id!r} has {n_in} row(s); at least {MIN_OBSERVATIONS} are needed to infer a frequency.",
                series_id=series_id,
                n_rows=n_in,
            )
        values = _parse_values(part["_raw"], source, series_id)
        index = pd.DatetimeIndex(part[TIMESTAMP_COLUMN])
        freq = infer_frequency(index)
        n_gap_filled = 0
        if freq.kind == "irregular":
            _fail(
                "FREQUENCY_IRREGULAR",
                source,
                f"series {series_id!r} is not regularly sampled (no single interval and no calendar interval).",
                series_id=series_id,
            )
        if freq.kind == "gapped":
            if config.gap_policy == "refuse":
                _fail(
                    "FREQUENCY_GAP",
                    source,
                    f"series {series_id!r} skips {freq.n_gaps} grid point(s) at interval {freq.alias}, first after "
                    f"{freq.first_gap_after}; fill them or set gap_policy='fill'.",
                    series_id=series_id,
                    n_gaps=freq.n_gaps,
                    first_gap_after=str(freq.first_gap_after),
                )
            full = pd.date_range(index[0], index[-1], freq=freq.step)
            filled = pd.Series(values, index=index).reindex(full)
            n_gap_filled = int(len(full) - len(index))
            values = filled.to_numpy(dtype=float)
            index = full
            changes.append(f"{series_id}: inserted {n_gap_filled} missing grid point(s) (gap_policy='fill')")
            freq = FrequencyInfo("fixed", freq.alias, freq.step)
        values, n_lead, n_interp = _apply_missing_policy(values, config, source, series_id)
        if n_lead:
            index = index[n_lead:]
            changes.append(f"{series_id}: dropped {n_lead} leading missing value(s)")
        if n_interp:
            changes.append(f"{series_id}: linearly interpolated {n_interp} missing value(s)")
        if len(values) < needed:
            _fail(
                "MIN_HISTORY",
                source,
                f"series {series_id!r} has {len(values)} usable row(s); horizon {config.horizon} + minimum context "
                f"{MIN_CONTEXT_POINTS} requires at least {needed}.",
                series_id=series_id,
                n_rows=len(values),
                required=needed,
            )
        truncated_from: int | None = None
        if len(values) > MAX_CONTEXT_POINTS:
            truncated_from = len(values)
            values = values[-MAX_CONTEXT_POINTS:]
            index = index[-MAX_CONTEXT_POINTS:]
            changes.append(
                f"{series_id}: kept the most recent {MAX_CONTEXT_POINTS} of {truncated_from} rows (model context limit)"
            )
        frequencies[series_id] = freq
        reports[series_id] = SeriesReport(
            series_id=series_id,
            n_rows_in=n_in,
            n_rows_out=len(values),
            start=str(index[0]),
            end=str(index[-1]),
            n_leading_missing_dropped=n_lead,
            n_interpolated=n_interp,
            n_gap_points_filled=n_gap_filled,
            truncated_from=truncated_from,
        )
        parts.append(
            pd.DataFrame({ID_COLUMN: series_id, TIMESTAMP_COLUMN: index, VALUE_COLUMN: values.astype(float)})
        )

    aliases = {f.alias for f in frequencies.values()}
    if len(aliases) > 1:
        _fail(
            "FREQUENCY_MIXED",
            source,
            f"series have different sampling intervals: { {k: v.alias for k, v in frequencies.items()} }.",
            aliases=sorted(a for a in aliases if a),
        )
    out = pd.concat(parts, ignore_index=True)
    return ValidationResult(
        frame=out,
        frequency=next(iter(frequencies.values())),
        series=reports,
        source=source,
        config=config,
        changes=changes,
    )

**Module 4/7:** `src/timesfm_forecasting/model.py` (carried verbatim; see the note above)

In [ ]:
"""Pinned TimesFM 3.0 checkpoint: identity, staging, integrity and loading.

Only ``google/timesfm-3.0-pytorch`` is accepted. Its files are listed in the committed manifest
(``weights/timesfm-3.0-pytorch/dimer-base-manifest.json``) with byte sizes and, where established,
SHA-256 digests. ``stage_missing_files`` fetches absent entries from the Hugging Face Hub,
``verify_snapshot`` re-checks every file against the manifest, and ``load_pinned_model`` loads the
verified snapshot directory through the ``timesfm3`` package of the pinned ``timesfm`` distribution
(``TimesFM3Forecaster``, which reads the staged ``config.json`` and ``model.safetensors`` and never
consults the Hub). Pickle-format weights are refused.

Weights licence
---------------
The TimesFM 3.0 weights are distributed under the *TimesFM Non-Commercial License v1.0*
(``license: other`` on the Hub, ``LICENSE`` in the checkpoint repository): non-commercial and
non-production use only, no redistribution of the model or of derivatives. The pipeline code is
Apache-2.0; the weights are downloaded at run time and never committed. ``MODEL_LICENSE_TERMS``
carries the restriction into every provenance record.

Revision status
---------------
``MODEL_REVISION`` is the immutable commit ``43046b85ec22d584a13f8098c2ed39c889e129c2``: the commit the Hub
served for ``google/timesfm-3.0-pytorch`` at ref ``main`` on the maintainer's hosted Colab run of 2026-10-06
(``docs/execution-evidence/2026-10-06/timesfm_forecasting_colab_86f2213_run2_passed.ipynb``, recorded in
``docs/release-verification.md``). That run also computed the SHA-256 of the 1,322,898,824-byte
``model.safetensors`` it staged; the digest is pinned in the manifest and ``verify_snapshot`` now enforces it
(size and digest) for every manifest entry. The file *names* and *byte sizes* and the digest of
``config.json`` were verified against the Hub's file listing and file contents when the pin was first
written (see the manifest's ``provenance`` block). ``stage_missing_files`` still writes
``resolved-revision.json`` and refuses a snapshot for which the Hub reported a commit other than the pinned
one. ``torch`` and ``timesfm3`` are imported lazily so that the rest of the package works without them.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .config import MAX_CONTEXT_POINTS, MAX_HORIZON, MEDIAN_SLOT, TRAINED_QUANTILES, ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import HubUnavailableError, ModelIntegrityError, ModelSourceError` removed — names are kernel globals defined by the carried modules

__all__ = [
    "MODEL_ID",
    "MODEL_REVISION",
    "MODEL_REVISION_STATUS",
    "MODEL_LICENSE",
    "MODEL_LICENSE_SOURCE",
    "MODEL_LICENSE_TERMS",
    "MODEL_LICENSE_URL",
    "MODEL_KEY",
    "MODEL_URL",
    "MANIFEST_NAME",
    "DEFAULT_WEIGHTS_DIR",
    "WEIGHTS_FILENAME",
    "CONFIG_FILENAME",
    "PINNED_WEIGHTS_BYTES",
    "PINNED_WEIGHTS_SHA256",
    "PINNED_CONFIG_SHA256",
    "REFUSED_WEIGHT_SUFFIXES",
    "EXPECTED_CONFIG",
    "EXPECTED_TRANSFORMER",
    "DECODE_SETTINGS",
    "LOAD_SETTINGS",
    "LoadedModel",
    "check_model_source",
    "read_manifest",
    "stage_missing_files",
    "verify_snapshot",
    "load_pinned_model",
    "sha256_file",
]

MODEL_ID = "google/timesfm-3.0-pytorch"
#: Immutable commit of the checkpoint repository: the commit the Hub served for ref ``main`` on the hosted run below.
MODEL_REVISION = "43046b85ec22d584a13f8098c2ed39c889e129c2"
MODEL_REVISION_STATUS = (
    "pinned: the commit the Hub served for google/timesfm-3.0-pytorch at ref main on the maintainer's hosted Colab run "
    "of 2026-10-06 (docs/execution-evidence/2026-10-06/timesfm_forecasting_colab_86f2213_run2_passed.ipynb); the "
    "model.safetensors SHA-256 observed on the same run is pinned in the manifest and enforced by verify_snapshot"
)
#: The Hub front matter says ``license: other`` with ``license_name: timesfm-non-commercial-license-v1.0``.
MODEL_LICENSE = "timesfm-non-commercial-license-v1.0"
MODEL_LICENSE_SOURCE = (
    "Hugging Face model card front matter of google/timesfm-3.0-pytorch: `license: other`, "
    "`license_name: timesfm-non-commercial-license-v1.0`, `license_link: LICENSE` (7,270-byte LICENSE file in "
    "the checkpoint repository, read in full on 2026-10-06)"
)
MODEL_LICENSE_TERMS = (
    "non-commercial and non-production use only (testing, evaluation and research not tied to commercial gain, "
    "production deployment or revenue generation); no use in end-user-facing or production systems; no "
    "distribution of the model or of derivatives; a commercial licence from Google LLC is required for any "
    "other use"
)
MODEL_LICENSE_URL = "https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE"
MODEL_KEY = "timesfm-3.0-pytorch"
MODEL_URL = f"https://huggingface.co/{MODEL_ID}"
MANIFEST_NAME = "dimer-base-manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
WEIGHTS_FILENAME = "model.safetensors"
CONFIG_FILENAME = "config.json"
#: Byte size of model.safetensors as listed by the Hub and as staged on the 2026-10-06 hosted run.
PINNED_WEIGHTS_BYTES = 1_322_898_824
#: SHA-256 of model.safetensors, computed in-run over the staged 1,322,898,824-byte file on the 2026-10-06 hosted run.
PINNED_WEIGHTS_SHA256 = "a7592b0a8432baee54483254e5647856911ce69e09d09a9bb65904b2d98f17da"
#: SHA-256 of the 1,273-byte config.json, computed from the Hub-served text (verified).
PINNED_CONFIG_SHA256 = "ff17bbc07b792c5a904cca265b8468579d736a4fe84981da25eb871b0a125bc6"
REFUSED_WEIGHT_SUFFIXES = frozenset({".bin", ".pt", ".pth", ".ckpt", ".pkl", ".pickle"})
#: Architecture facts the loader asserts after reading config.json (top-level keys).
EXPECTED_CONFIG = {
    "input_patch_len": 32,
    "output_patch_len": 64,
    "quantiles": list(TRAINED_QUANTILES),
    "use_variate_attention": True,
    "use_iterative_cpm_revin": True,
    "use_linear_detrending": True,
    "use_stitching": True,
}
#: Architecture facts asserted under ``transformer_config`` (``num_layers``) and ``transformer_config.transformer``.
EXPECTED_TRANSFORMER = {"num_layers": 20, "model_dims": 1280, "num_heads": 16, "use_rope_var": False}
#: Upstream ``TimesFM3Forecaster.predict_batch`` options the pipeline passes on every call. ``sort_quantiles``
#: makes the nine slots monotone (the upstream default); the three transforms are off so the output is the
#: model's own quantile head, and the batch size is the upstream default.
# Constructor settings (``_ModelConfig`` fields of ``timesfm3.TimesFM3Forecaster``).
LOAD_SETTINGS = {
    "per_core_batch_size": 4,
}

# Keyword arguments of ``TimesFM3Forecaster.predict_batch``; nothing else may appear here.
DECODE_SETTINGS = {
    "return_quantiles": True,
    "sort_quantiles": True,
    "use_symmetric_averaging": False,
    "make_positive": False,
    "use_znorm": False,
    "padding_mode": "none",
}

_INPUT_PATCH = 32
_OUTPUT_PATCH = 64
_N_SLOTS = len(TRAINED_QUANTILES)


def _is_digest(value: Any) -> bool:
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def sha256_file(path: str | os.PathLike[str], *, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def check_model_source(model_id: Any, revision: Any) -> None:
    """Refuse any identity other than the pinned one (local paths, URIs, other repos, other refs)."""
    if model_id != MODEL_ID:
        raise ModelSourceError(
            f"Only the pinned checkpoint {MODEL_ID!r} is accepted; got {model_id!r}. Local paths, "
            "URIs and other repositories are refused."
        )
    if revision != MODEL_REVISION:
        raise ModelSourceError(
            f"Only the pinned revision {MODEL_REVISION!r} ({MODEL_REVISION_STATUS.split(':')[0]}) is "
            f"accepted; got {revision!r}."
        )


def read_manifest(root: str | os.PathLike[str] | None = None) -> dict[str, Any]:
    root_path = Path(root) if root is not None else DEFAULT_WEIGHTS_DIR
    path = root_path / MANIFEST_NAME
    if not path.is_file():
        raise ModelIntegrityError(f"manifest missing: {path}")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ModelIntegrityError(
            f"manifest at {path} names {manifest.get('modelId')!r}@{manifest.get('revision')!r}, "
            f"not the pinned {MODEL_ID!r}@{MODEL_REVISION!r}"
        )
    return manifest


def _hub_download(relative_path: str, root: Path) -> dict[str, Any]:
    """Fetch one manifest entry from the Hub into ``root``; returns what the Hub said it served."""
    try:
        from huggingface_hub import hf_hub_download
    except ImportError as exc:  # pragma: no cover - runtime dependency
        raise HubUnavailableError("huggingface_hub is not installed") from exc
    try:
        local = hf_hub_download(
            repo_id=MODEL_ID, filename=relative_path, revision=MODEL_REVISION, local_dir=str(root)
        )
    except Exception as exc:  # noqa: BLE001 - every hub failure is reported the same way
        raise HubUnavailableError(f"could not fetch {relative_path} from {MODEL_URL}: {exc}") from exc
    resolved: str | None = None
    try:
        from huggingface_hub import HfApi

        resolved = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    except Exception:  # noqa: BLE001 - best effort provenance only
        resolved = None
    return {"path": relative_path, "local": str(local), "resolved_revision": resolved}


def stage_missing_files(
    path: str | os.PathLike[str] | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], Any] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent from the snapshot directory.

    Returns the relative paths fetched. Also writes ``resolved-revision.json`` next to the
    manifest with whatever the downloader reported, so a hosted run leaves the commit it
    actually served on disk. The files are requested at the pinned commit, so a downloader that
    reports a different commit is a mismatch: the record flags it and ``ModelIntegrityError`` is
    raised. Downloading the weights accepts the non-commercial licence recorded in
    ``MODEL_LICENSE_TERMS``; the record repeats it.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them from "
            f"{MODEL_URL} at {MODEL_REVISION!r} (weights licence: {MODEL_LICENSE}, {MODEL_LICENSE_TERMS})"
        )
    fetch = downloader or _hub_download
    records = []
    for relative_path in missing:
        records.append(fetch(relative_path, root))
    resolved = [r.get("resolved_revision") for r in records if isinstance(r, dict)]
    served = next((r for r in resolved if r), None)
    mismatch = served is not None and served != MODEL_REVISION
    (root / "resolved-revision.json").write_text(
        json.dumps(
            {
                "modelId": MODEL_ID,
                "requested_revision": MODEL_REVISION,
                "revision_status": MODEL_REVISION_STATUS,
                "resolved_revision": served,
                "resolved_revision_matches_pin": None if served is None else not mismatch,
                "fetched": missing,
                "license": MODEL_LICENSE,
                "license_terms": MODEL_LICENSE_TERMS,
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    if mismatch:
        raise ModelIntegrityError(
            f"the Hub reported commit {served!r} for {MODEL_ID} but the pinned revision is {MODEL_REVISION!r}; "
            f"the snapshot at {root} is not accepted (see resolved-revision.json)"
        )
    return missing


def _config_mismatches(config: dict[str, Any]) -> list[str]:
    """Pinned architecture values that the staged ``config.json`` does not carry."""
    out = []
    for key, value in EXPECTED_CONFIG.items():
        if config.get(key) != value:
            out.append(f"{key}={config.get(key)!r} (pinned {value!r})")
    transformer = config.get("transformer_config") or {}
    inner = transformer.get("transformer") or {} if isinstance(transformer, dict) else {}
    for key, value in EXPECTED_TRANSFORMER.items():
        got = transformer.get(key) if key == "num_layers" else inner.get(key)
        if got != value:
            out.append(f"transformer_config.{key}={got!r} (pinned {value!r})")
    return out


def verify_snapshot(snapshot_path: str | os.PathLike[str]) -> dict[str, Any]:
    """Check every manifest entry by byte size and by SHA-256 against the manifest.

    Every entry must carry a 64-hex digest; a manifest that does not pin one (the former
    ``"pending: ..."`` sentinel) is refused rather than reported, so a snapshot is never loaded on a
    size check alone. Pickle-format weight files anywhere in the directory are refused.
    """
    root = Path(snapshot_path)
    if not root.is_dir():
        raise ModelIntegrityError(f"snapshot path is not a directory: {root}")
    manifest = read_manifest(root)
    refused = sorted(p.name for p in root.rglob("*") if p.is_file() and p.suffix.lower() in REFUSED_WEIGHT_SUFFIXES)
    if refused:
        raise ModelIntegrityError(
            f"refusing to load: pickle-format weight files {refused} are present in {root}; only "
            f"{WEIGHTS_FILENAME} is acceptable"
        )
    files = []
    for entry in manifest["files"]:
        target = root / entry["path"]
        if not target.is_file():
            raise ModelIntegrityError(f"missing {entry['path']} in snapshot {root}")
        actual_bytes = target.stat().st_size
        if actual_bytes != int(entry["bytes"]):
            raise ModelIntegrityError(
                f"{entry['path']} is {actual_bytes} bytes, expected {entry['bytes']} (manifest)"
            )
        expected = entry.get("sha256")
        if not _is_digest(expected):
            raise ModelIntegrityError(
                f"manifest carries no SHA-256 digest for {entry['path']} ({expected!r}); every staged file must be "
                "digest-verified"
            )
        observed = sha256_file(target)
        if observed != expected:
            raise ModelIntegrityError(
                f"{entry['path']} SHA-256 {observed} does not match the manifest digest {expected}"
            )
        files.append({"path": entry["path"], "bytes": actual_bytes, "sha256": observed, "verified": True})
    if (root / CONFIG_FILENAME).is_file():
        config = json.loads((root / CONFIG_FILENAME).read_text(encoding="utf-8"))
        mismatches = _config_mismatches(config)
        if mismatches:
            raise ModelIntegrityError(
                f"{CONFIG_FILENAME} differs from the pinned architecture: {'; '.join(mismatches)}; this is not "
                "the TimesFM 3.0 PyTorch checkpoint"
            )
    resolved_file = root / "resolved-revision.json"
    resolved = None
    if resolved_file.is_file():
        resolved = json.loads(resolved_file.read_text(encoding="utf-8")).get("resolved_revision")
    return {
        "modelId": MODEL_ID,
        "revision": MODEL_REVISION,
        "revision_status": MODEL_REVISION_STATUS,
        "resolved_revision": resolved,
        "license": MODEL_LICENSE,
        "files": files,
        "totalBytes": sum(int(f["bytes"]) for f in files),
    }


@dataclass
class LoadedModel:
    """The pinned TimesFM 3.0 forecaster: one upstream ``TimesFM3Forecaster`` and its identity record."""

    model: Any
    identity: dict[str, Any]
    device: str
    dtype: str
    source: str
    snapshot: dict[str, Any]

    @property
    def trained_quantiles(self) -> tuple[float, ...]:
        return TRAINED_QUANTILES

    def decode_settings(self, config: ForecastConfig) -> dict[str, Any]:
        """The upstream options one ``predict`` call uses, plus the patch envelope the request rounds to.

        TimesFM 3.0 needs no compile step: the forecaster pads each batch's context up to a multiple of
        the 32-point input patch (capped at the 15,360-point context limit) and decodes the horizon in
        64-point output patches, so these values are recorded for provenance, not applied by the pipeline.
        """
        return {
            **LOAD_SETTINGS,
            **DECODE_SETTINGS,
            "context_patch_envelope": min(-(-config.context_length // _INPUT_PATCH) * _INPUT_PATCH, MAX_CONTEXT_POINTS),
            "horizon_patch_envelope": -(-config.horizon // _OUTPUT_PATCH) * _OUTPUT_PATCH,
            "median_quantile_index": MEDIAN_SLOT,
            "input_patch_len": _INPUT_PATCH,
            "output_patch_len": _OUTPUT_PATCH,
            "context_limit": MAX_CONTEXT_POINTS,
        }

    def predict(self, inputs: list[np.ndarray], horizon: int, config: ForecastConfig) -> tuple[np.ndarray, np.ndarray]:
        """Zero-shot forecast: ``(point, quantiles)`` of shapes ``(n, horizon)`` and ``(n, horizon, 9)``.

        ``quantiles[..., k]`` is the model's quantile ``TRAINED_QUANTILES[k]`` (0.1 .. 0.9) and ``point``
        is its median, ``quantiles[..., MEDIAN_SLOT]`` (slot 4). There is no separate mean head in
        TimesFM 3.0.
        """
        if horizon > MAX_HORIZON:
            raise ValueError(f"horizon {horizon} exceeds MAX_HORIZON={MAX_HORIZON}")
        arrays = [np.asarray(x, dtype=np.float32) for x in inputs]
        outputs = list(self.model.predict_batch(contexts=arrays, horizon=horizon, **DECODE_SETTINGS))
        if len(outputs) != len(arrays):
            raise ModelIntegrityError(f"upstream returned {len(outputs)} forecasts for {len(arrays)} series")
        point = np.asarray(np.stack([np.asarray(o.forecast, dtype=float) for o in outputs]), dtype=float)
        quantiles = np.asarray(np.stack([np.asarray(o.quantiles, dtype=float) for o in outputs]), dtype=float)
        if point.shape != (len(arrays), horizon) or quantiles.shape != (len(arrays), horizon, _N_SLOTS):
            raise ModelIntegrityError(
                f"upstream returned shapes {point.shape} / {quantiles.shape}; expected "
                f"({len(arrays)}, {horizon}) / ({len(arrays)}, {horizon}, {_N_SLOTS})"
            )
        if not (np.isfinite(point).all() and np.isfinite(quantiles).all()):
            raise ModelIntegrityError("upstream returned a non-finite forecast value")
        return point, quantiles


def resolve_device(device: str) -> str:
    if device != "auto":
        return device
    import torch

    return "cuda" if torch.cuda.is_available() else "cpu"


def load_pinned_model(
    weights_dir: str | os.PathLike[str] | None = None,
    *,
    device: str = "auto",
) -> LoadedModel:
    """Verify the staged snapshot, then load it through ``timesfm3.TimesFM3Forecaster``.

    The loader never consults the Hub for the identity: the committed manifest is the anchor. The
    forecaster is given the snapshot *directory*, so it builds the model from the staged
    ``config.json`` (the same path upstream's ``from_pretrained`` takes) rather than from the
    package's built-in defaults, and ``local_files_only=True`` forbids any download at load time.
    """
    root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
    snapshot = verify_snapshot(root)
    import torch
    from timesfm3 import TimesFM3Forecaster

    resolved_device = resolve_device(device)
    if resolved_device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("device='cuda' requested but CUDA is not available")
    model = TimesFM3Forecaster(
        checkpoint_path=str(root),
        device=resolved_device,
        local_files_only=True,
        **LOAD_SETTINGS,
    )
    forecaster_config = getattr(model, "config", None)
    if forecaster_config is not None:
        observed = {
            "input_patch_length": getattr(forecaster_config, "input_patch_length", None),
            "output_patch_length": getattr(forecaster_config, "output_patch_length", None),
            "quantiles": list(getattr(forecaster_config, "quantiles", []) or []),
            "median_quantile_index": getattr(forecaster_config, "median_quantile_index", None),
        }
        expected = {
            "input_patch_length": _INPUT_PATCH,
            "output_patch_length": _OUTPUT_PATCH,
            "quantiles": list(TRAINED_QUANTILES),
            "median_quantile_index": MEDIAN_SLOT,
        }
        if observed != expected:
            raise ModelIntegrityError(f"the loaded forecaster reports {observed}, not the pinned {expected}")
    param = next(model.model.parameters())
    weight_entry = next(f for f in snapshot["files"] if f["path"] == WEIGHTS_FILENAME)
    identity = {
        "model_id": MODEL_ID,
        "revision": MODEL_REVISION,
        "revision_status": MODEL_REVISION_STATUS,
        "resolved_revision": snapshot.get("resolved_revision"),
        "license": MODEL_LICENSE,
        "license_terms": MODEL_LICENSE_TERMS,
        "license_url": MODEL_LICENSE_URL,
        "license_source": MODEL_LICENSE_SOURCE,
        "source_url": MODEL_URL,
        "weights_file": WEIGHTS_FILENAME,
        "weights_bytes": weight_entry["bytes"],
        "weights_sha256": weight_entry["sha256"],
        "weights_sha256_verified_against_manifest": weight_entry["verified"],
        "config_sha256": PINNED_CONFIG_SHA256,
        "trained_quantiles": list(TRAINED_QUANTILES),
        "median_quantile_index": MEDIAN_SLOT,
        "parameters": int(sum(p.numel() for p in model.model.parameters())),
    }
    return LoadedModel(
        model=model,
        identity=identity,
        device=str(param.device),
        dtype=str(param.dtype),
        source="local-snapshot (manifest-anchored)",
        snapshot=snapshot,
    )

**Module 5/7:** `src/timesfm_forecasting/forecasting.py` (carried verbatim; see the note above)

In [ ]:
"""Zero-shot forecasting over a validated table: point (median) and quantile output.

``forecast`` takes a :class:`ValidationResult`, passes each series' most recent
``context_length`` points to the model, and returns one tidy row per ``(series_id, step)`` with
the future timestamp, the median point forecast and the requested quantiles. TimesFM 3.0 emits
nine quantile slots (the deciles 0.1 .. 0.9) and no mean head; the point forecast is slot 4, the
median. The row layout is asserted against the model's output shape so a forecast can never be
attached to the wrong series or step.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import MEDIAN_SLOT, TRAINED_QUANTILES, ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .data import ID_COLUMN, TIMESTAMP_COLUMN, VALUE_COLUMN, ValidationResult, future_timestamps` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import ModelIntegrityError` removed — names are kernel globals defined by the carried modules

__all__ = ["ForecastResult", "forecast", "quantile_column", "POINT_COLUMN", "STEP_COLUMN"]

POINT_COLUMN = "prediction"
STEP_COLUMN = "step"


def quantile_column(level: float) -> str:
    return f"q{level:g}"


@dataclass(frozen=True)
class ForecastResult:
    """A forecast frame plus the inference provenance that produced it."""

    forecast: pd.DataFrame
    provenance: dict[str, Any]

    @property
    def quantile_columns(self) -> list[str]:
        return [c for c in self.forecast.columns if c.startswith("q")]


def _series_arrays(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    return {str(k): g for k, g in frame.groupby(ID_COLUMN, sort=False)}


def forecast(validation: ValidationResult, config: ForecastConfig, model: Any) -> ForecastResult:
    """Forecast every series in ``validation.frame`` ``config.horizon`` steps ahead."""
    groups = _series_arrays(validation.frame)
    ids = list(groups)
    inputs: list[np.ndarray] = []
    effective_context: dict[str, int] = {}
    for series_id in ids:
        values = groups[series_id][VALUE_COLUMN].to_numpy(dtype=float)
        window = values[-config.context_length :]
        effective_context[series_id] = int(len(window))
        inputs.append(window)
    point, quantiles = model.predict(inputs, config.horizon, config)
    if point.shape != (len(ids), config.horizon) or quantiles.shape[:2] != (len(ids), config.horizon):
        raise ModelIntegrityError(f"model returned {point.shape} / {quantiles.shape} for {len(ids)} series")
    if quantiles.shape[2] != len(TRAINED_QUANTILES):
        raise ModelIntegrityError(f"expected {len(TRAINED_QUANTILES)} quantile slots, got {quantiles.shape[2]}")
    # Row-layout oracle: the point forecast must equal the median slot (index 4) for every series/step.
    if not np.allclose(point, quantiles[:, :, MEDIAN_SLOT], rtol=0, atol=1e-6):
        raise ModelIntegrityError("point forecast is not the median slot; the upstream contract changed")
    rows = []
    for i, series_id in enumerate(ids):
        last = pd.Timestamp(groups[series_id][TIMESTAMP_COLUMN].iloc[-1])
        stamps = future_timestamps(last, validation.frequency, config.horizon)
        for step in range(config.horizon):
            row = {
                ID_COLUMN: series_id,
                TIMESTAMP_COLUMN: stamps[step],
                STEP_COLUMN: step + 1,
                POINT_COLUMN: float(point[i, step]),
            }
            for level in config.quantile_levels:
                slot = int(round(level * 10)) - 1  # 0.1 -> slot 0 ... 0.9 -> slot 8
                row[quantile_column(level)] = float(quantiles[i, step, slot])
            rows.append(row)
    out = pd.DataFrame(rows)
    provenance = {
        "task": "zero-shot time-series forecasting",
        "point_forecast_semantics": "median (model quantile 0.5, slot 4 of the nine quantile slots); no mean head",
        "quantile_semantics": (
            "model quantiles from the trained decile grid, sorted monotone upstream; not calibrated intervals"
        ),
        "horizon": config.horizon,
        "requested_context_length": config.context_length,
        "effective_context_length": effective_context,
        "quantile_levels": list(config.quantile_levels),
        "n_series": len(ids),
        "frequency": validation.frequency.to_dict(),
        "decode_settings": model.decode_settings(config) if hasattr(model, "decode_settings") else None,
    }
    return ForecastResult(forecast=out, provenance=provenance)

**Module 6/7:** `src/timesfm_forecasting/evaluation.py` (carried verbatim; see the note above)

In [ ]:
"""Chronological evaluation, naive and statistical baselines, and forecasting metrics.

The only split helper is chronological: the last ``horizon`` points of every series are held
out as truth and removed from the history the model sees. Baselines use history only. Metrics
are MASE (scaled by the in-sample seasonal-naive error), sMAPE and the quantile (pinball) loss.
``evaluation_report`` records verdicts; it never asserts that the model beats a baseline.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .data import ID_COLUMN, TIMESTAMP_COLUMN, VALUE_COLUMN, FrequencyInfo, future_timestamps` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import ValidationError` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .forecasting import POINT_COLUMN, STEP_COLUMN` removed — names are kernel globals defined by the carried modules

__all__ = [
    "HoldoutSplit",
    "chronological_holdout",
    "naive_baseline",
    "seasonal_naive_baseline",
    "ses_baseline",
    "mase",
    "smape",
    "quantile_loss",
    "evaluate_forecast",
    "evaluation_report",
    "BASELINE_NAMES",
]

BASELINE_NAMES = ("naive", "seasonal_naive", "ses")


@dataclass(frozen=True)
class HoldoutSplit:
    history: pd.DataFrame
    truth: pd.DataFrame
    horizon: int

    def leakage_check(self) -> dict[str, Any]:
        """Per series: the last history timestamp precedes the first truth timestamp."""
        out = {}
        for series_id, g in self.truth.groupby(ID_COLUMN, sort=False):
            h = self.history[self.history[ID_COLUMN] == series_id]
            out[str(series_id)] = bool(h[TIMESTAMP_COLUMN].max() < g[TIMESTAMP_COLUMN].min())
        return out


def chronological_holdout(frame: pd.DataFrame, horizon: int) -> HoldoutSplit:
    """Hold out the final ``horizon`` rows of every series as truth."""
    if horizon < 1:
        raise ValueError("horizon must be positive")
    histories, truths = [], []
    for series_id, g in frame.groupby(ID_COLUMN, sort=False):
        g = g.sort_values(TIMESTAMP_COLUMN, kind="mergesort")
        if len(g) <= horizon:
            raise ValidationError(
                "MIN_HISTORY",
                "<holdout>",
                f"series {series_id!r} has {len(g)} rows; a {horizon}-step holdout leaves no history.",
                {"series_id": series_id, "n_rows": len(g), "horizon": horizon},
            )
        histories.append(g.iloc[:-horizon])
        truths.append(g.iloc[-horizon:])
    return HoldoutSplit(
        history=pd.concat(histories, ignore_index=True),
        truth=pd.concat(truths, ignore_index=True),
        horizon=horizon,
    )


# ---------------------------------------------------------------------------
# Baselines (history only)
# ---------------------------------------------------------------------------


def _baseline_frame(
    history: pd.DataFrame, frequency: FrequencyInfo, horizon: int, fn, name: str
) -> pd.DataFrame:
    rows = []
    for series_id, g in history.groupby(ID_COLUMN, sort=False):
        g = g.sort_values(TIMESTAMP_COLUMN, kind="mergesort")
        values = g[VALUE_COLUMN].to_numpy(dtype=float)
        preds = np.asarray(fn(values), dtype=float)
        if preds.shape != (horizon,):
            raise RuntimeError(f"{name}: expected {horizon} predictions, got {preds.shape}")
        stamps = future_timestamps(pd.Timestamp(g[TIMESTAMP_COLUMN].iloc[-1]), frequency, horizon)
        for step in range(horizon):
            rows.append(
                {
                    ID_COLUMN: series_id,
                    TIMESTAMP_COLUMN: stamps[step],
                    STEP_COLUMN: step + 1,
                    POINT_COLUMN: float(preds[step]),
                    "method": name,
                }
            )
    return pd.DataFrame(rows)


def naive_baseline(history: pd.DataFrame, frequency: FrequencyInfo, horizon: int) -> pd.DataFrame:
    """Repeat the last observed value."""
    return _baseline_frame(history, frequency, horizon, lambda v: np.repeat(v[-1], horizon), "naive")


def seasonal_naive_baseline(
    history: pd.DataFrame, frequency: FrequencyInfo, horizon: int, season_length: int
) -> pd.DataFrame:
    """Repeat the last full season; refuses a series shorter than one season."""
    if isinstance(season_length, bool) or not isinstance(season_length, int) or season_length < 1:
        raise ValueError(f"season_length must be a positive int, got {season_length!r}")

    def fn(v: np.ndarray) -> np.ndarray:
        if len(v) < season_length:
            raise ValidationError(
                "SEASON_TOO_LONG",
                "<history>",
                f"a series has {len(v)} rows, fewer than season_length={season_length}.",
                {"n_rows": len(v), "season_length": season_length},
            )
        season = v[-season_length:]
        return np.array([season[k % season_length] for k in range(horizon)])

    return _baseline_frame(history, frequency, horizon, fn, "seasonal_naive")


def _ses_fit(values: np.ndarray, alphas: np.ndarray) -> tuple[float, float]:
    """Simple exponential smoothing: choose alpha by in-sample one-step SSE; return (alpha, level)."""
    best_alpha, best_sse, best_level = 0.5, np.inf, float(values[-1])
    for alpha in alphas:
        level = float(values[0])
        sse = 0.0
        for x in values[1:]:
            sse += (x - level) ** 2
            level = alpha * x + (1 - alpha) * level
        if sse < best_sse:
            best_alpha, best_sse, best_level = float(alpha), sse, level
    return best_alpha, best_level


def ses_baseline(history: pd.DataFrame, frequency: FrequencyInfo, horizon: int) -> pd.DataFrame:
    """Simple exponential smoothing (flat forecast at the fitted level), alpha chosen in-sample."""
    alphas = np.linspace(0.05, 0.95, 19)
    return _baseline_frame(history, frequency, horizon, lambda v: np.repeat(_ses_fit(v, alphas)[1], horizon), "ses")


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------


def mase(y_true: np.ndarray, y_pred: np.ndarray, insample: np.ndarray, season_length: int = 1) -> float:
    """Mean absolute scaled error: MAE over the in-sample seasonal-naive MAE (Hyndman & Koehler)."""
    y_true, y_pred, insample = (np.asarray(a, dtype=float) for a in (y_true, y_pred, insample))
    if len(insample) <= season_length:
        return float("nan")
    scale = np.mean(np.abs(insample[season_length:] - insample[:-season_length]))
    if scale == 0:
        return float("nan")
    return float(np.mean(np.abs(y_true - y_pred)) / scale)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric MAPE in percent; a term with zero denominator counts as zero error."""
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(denom == 0, 0.0, np.abs(y_true - y_pred) / denom)
    return float(100 * np.mean(terms))


def quantile_loss(y_true: np.ndarray, y_q: np.ndarray, level: float) -> float:
    """Pinball loss at ``level``: mean of ``max(level*(y-q), (level-1)*(y-q))``."""
    y_true, y_q = np.asarray(y_true, dtype=float), np.asarray(y_q, dtype=float)
    diff = y_true - y_q
    return float(np.mean(np.maximum(level * diff, (level - 1) * diff)))


def _aligned(forecast_df: pd.DataFrame, truth: pd.DataFrame) -> pd.DataFrame:
    keys = [ID_COLUMN, TIMESTAMP_COLUMN]
    f = forecast_df.copy()
    t = truth[[*keys, VALUE_COLUMN]].copy()
    f[TIMESTAMP_COLUMN] = pd.to_datetime(f[TIMESTAMP_COLUMN])
    t[TIMESTAMP_COLUMN] = pd.to_datetime(t[TIMESTAMP_COLUMN])
    merged = f.merge(t, on=keys, how="outer", indicator=True)
    bad = merged[merged["_merge"] != "both"]
    if len(bad):
        raise ValidationError(
            "EVALUATION_MISALIGNED",
            "<evaluation>",
            f"{len(bad)} forecast/truth rows do not line up on (series_id, timestamp); first: "
            f"{bad.iloc[0][ID_COLUMN]!r} at {bad.iloc[0][TIMESTAMP_COLUMN]} ({bad.iloc[0]['_merge']}).",
            {"n_misaligned": int(len(bad))},
        )
    return merged.drop(columns="_merge")


def evaluate_forecast(
    forecast_df: pd.DataFrame,
    truth: pd.DataFrame,
    history: pd.DataFrame,
    config: ForecastConfig,
) -> dict[str, Any]:
    """Per-series and pooled metrics of ``forecast_df`` against ``truth``.

    Rows must match exactly on ``(series_id, timestamp)``; a mismatch is refused rather than
    dropped. MASE is scaled by ``config.season_length`` (or 1).
    """
    aligned = _aligned(forecast_df, truth)
    m = config.season_length or 1
    q_cols = [c for c in forecast_df.columns if c.startswith("q") and c[1:].replace(".", "", 1).isdigit()]
    per_series = {}
    for series_id, g in aligned.groupby(ID_COLUMN, sort=False):
        g = g.sort_values(TIMESTAMP_COLUMN, kind="mergesort")
        insample = history[history[ID_COLUMN] == series_id].sort_values(TIMESTAMP_COLUMN)[VALUE_COLUMN].to_numpy(float)
        y, yhat = g[VALUE_COLUMN].to_numpy(float), g[POINT_COLUMN].to_numpy(float)
        entry: dict[str, Any] = {
            "n": int(len(g)),
            "mae": float(np.mean(np.abs(y - yhat))),
            "mase": mase(y, yhat, insample, m),
            "smape": smape(y, yhat),
        }
        if q_cols:
            entry["quantile_loss"] = {c: quantile_loss(y, g[c].to_numpy(float), float(c[1:])) for c in q_cols}
            entry["mean_quantile_loss"] = float(np.mean(list(entry["quantile_loss"].values())))
            lo, hi = min(q_cols, key=lambda c: float(c[1:])), max(q_cols, key=lambda c: float(c[1:]))
            if lo != hi:
                inside = (y >= g[lo].to_numpy(float)) & (y <= g[hi].to_numpy(float))
                entry["interval"] = {"lower": lo, "upper": hi, "nominal": round(float(hi[1:]) - float(lo[1:]), 2)}
                entry["interval_coverage"] = float(np.mean(inside))
        per_series[str(series_id)] = entry
    pooled: dict[str, Any] = {
        "n_series": len(per_series),
        "n_points": int(sum(e["n"] for e in per_series.values())),
        "mae": float(np.mean([e["mae"] for e in per_series.values()])),
        "mase": float(np.nanmean([e["mase"] for e in per_series.values()])),
        "smape": float(np.mean([e["smape"] for e in per_series.values()])),
        "mase_season_length": m,
    }
    if q_cols:
        pooled["mean_quantile_loss"] = float(np.mean([e["mean_quantile_loss"] for e in per_series.values()]))
        pooled["quantile_loss"] = {
            c: float(np.mean([e["quantile_loss"][c] for e in per_series.values()])) for c in q_cols
        }
        cov = [e["interval_coverage"] for e in per_series.values() if "interval_coverage" in e]
        if cov:
            pooled["interval_coverage"] = float(np.mean(cov))
            pooled["interval"] = next(iter(per_series.values()))["interval"]
    return {"estimation": "chronological tail holdout", "per_series": per_series, "aggregate": pooled}


def evaluation_report(
    model_eval: dict[str, Any],
    baseline_evals: dict[str, dict[str, Any]],
    *,
    horizon: int,
    sample_kind: str,
) -> dict[str, Any]:
    """Compare the model with each baseline and *record* verdicts (no assertion).

    ``verdict`` is ``sample-sanity``: one tail holdout on one small sample shows the pipeline
    works end to end and how the model compares on that sample; it is not benchmark evidence.
    """
    rows = [{"method": "timesfm", **{k: model_eval["aggregate"].get(k) for k in ("mae", "mase", "smape")}}]
    comparison = {}
    for name, ev in baseline_evals.items():
        agg = ev["aggregate"]
        rows.append({"method": name, **{k: agg.get(k) for k in ("mae", "mase", "smape")}})
        comparison[name] = {
            "model_mase_lower": bool(model_eval["aggregate"]["mase"] < agg["mase"])
            if np.isfinite(model_eval["aggregate"]["mase"]) and np.isfinite(agg["mase"])
            else None,
            "model_smape_lower": bool(model_eval["aggregate"]["smape"] < agg["smape"]),
        }
    beaten = [n for n, c in comparison.items() if c["model_mase_lower"]]
    return {
        "verdict": "sample-sanity",
        "reason": (
            f"one chronological holdout of {horizon} steps per series on {sample_kind}; metrics are tutorial "
            "evidence, not a benchmark"
        ),
        "horizon": horizon,
        "metrics": {"mase": "MAE / in-sample seasonal-naive MAE", "smape": "percent", "quantile_loss": "pinball"},
        "table": rows,
        "comparison": comparison,
        "model_beats_on_mase": beaten,
        "model_quantile_loss": model_eval["aggregate"].get("quantile_loss"),
        "model_interval_coverage": model_eval["aggregate"].get("interval_coverage"),
        "interval": model_eval["aggregate"].get("interval"),
    }

**Module 7/7:** `src/timesfm_forecasting/provenance.py` (carried verbatim; see the note above)

In [ ]:
"""Runtime provenance, the exported result bundle, and the reload-parity check.

The bundle written by :func:`write_result_bundle` is what a downstream reader consumes: a
forecast CSV, an evaluation JSON and ``result.json`` carrying every identity, digest, version
and configuration needed to interpret the forecast. :func:`check_reload_parity` reloads the
bundle from disk and compares it with the in-memory forecast; a mismatch is a contract failure
and raises.
"""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .data import ValidationResult` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .forecasting import ForecastResult` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import MODEL_ID, MODEL_LICENSE, MODEL_REVISION, MODEL_REVISION_STATUS` removed — names are kernel globals defined by the carried modules

__all__ = [
    "runtime_versions",
    "build_provenance",
    "write_result_bundle",
    "reload_result_bundle",
    "check_reload_parity",
    "sha256_bytes",
]


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


def runtime_versions() -> dict[str, Any]:
    out: dict[str, Any] = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "timesfm": _version("timesfm"),
        "torch": _version("torch"),
        "huggingface_hub": _version("huggingface-hub"),
        "safetensors": _version("safetensors"),
    }
    try:
        import torch

        out["cuda_available"] = bool(torch.cuda.is_available())
    except ImportError:
        out["cuda_available"] = None
    return out


def build_provenance(
    model: Any,
    config: ForecastConfig,
    validation: ValidationResult,
    result: ForecastResult,
    *,
    data_source: dict[str, Any],
    notebook_source: dict[str, Any] | None = None,
) -> dict[str, Any]:
    identity = dict(getattr(model, "identity", {}) or {})
    identity.setdefault("model_id", MODEL_ID)
    identity.setdefault("revision", MODEL_REVISION)
    identity.setdefault("revision_status", MODEL_REVISION_STATUS)
    identity.setdefault("license", MODEL_LICENSE)
    return {
        "model": identity,
        "device": getattr(model, "device", None),
        "dtype": getattr(model, "dtype", None),
        "weights_source": getattr(model, "source", None),
        "config": config.to_dict(),
        "inference": result.provenance,
        "input_manifest": validation.manifest(),
        "data_source": data_source,
        "runtime": runtime_versions(),
        "notebook_source": notebook_source,
        "determinism": (
            "CPU float32 inference is deterministic for fixed inputs and pins; CUDA kernels may differ "
            "in the last bits"
        ),
    }


def _json_ready(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): _json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_ready(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, (pd.Timestamp, pd.Timedelta)):
        return str(obj)
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    return obj


def write_result_bundle(
    out_dir: str | Path,
    stem: str,
    *,
    forecast: pd.DataFrame,
    evaluation: dict[str, Any] | None,
    report: dict[str, Any] | None,
    provenance: dict[str, Any],
    extra: dict[str, Any] | None = None,
) -> dict[str, str]:
    """Write ``<stem>_forecast.csv``, ``<stem>_evaluation.json``, ``<stem>_provenance.json`` and
    ``<stem>_result.json`` (which indexes the others with their digests) under ``out_dir``."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths: dict[str, str] = {}
    csv_path = out / f"{stem}_forecast.csv"
    forecast.to_csv(csv_path, index=False, lineterminator="\n", float_format="%.10g")
    paths["forecast_csv"] = str(csv_path)
    files = {"forecast_csv": csv_path}
    if evaluation is not None:
        p = out / f"{stem}_evaluation.json"
        p.write_text(json.dumps(_json_ready(evaluation), indent=2) + "\n", encoding="utf-8")
        paths["evaluation_json"] = str(p)
        files["evaluation_json"] = p
    p = out / f"{stem}_provenance.json"
    p.write_text(json.dumps(_json_ready(provenance), indent=2) + "\n", encoding="utf-8")
    paths["provenance_json"] = str(p)
    files["provenance_json"] = p
    result = {
        "format": "dimer-forecast-bundle",
        "format_version": 1,
        "stem": stem,
        "model": provenance.get("model"),
        "files": {
            k: {"path": v.name, "bytes": v.stat().st_size, "sha256": sha256_bytes(v.read_bytes())}
            for k, v in files.items()
        },
        "forecast_columns": list(forecast.columns),
        "n_rows": int(len(forecast)),
        "evaluation_report": report,
        "provenance_summary": {
            "horizon": provenance.get("config", {}).get("horizon"),
            "n_series": provenance.get("inference", {}).get("n_series"),
            "device": provenance.get("device"),
            "runtime": provenance.get("runtime"),
        },
        **(extra or {}),
    }
    rp = out / f"{stem}_result.json"
    rp.write_text(json.dumps(_json_ready(result), indent=2) + "\n", encoding="utf-8")
    paths["result_json"] = str(rp)
    return paths


def reload_result_bundle(out_dir: str | Path, stem: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Read the bundle back from files, checking each indexed file's digest first."""
    out = Path(out_dir)
    result = json.loads((out / f"{stem}_result.json").read_text(encoding="utf-8"))
    for entry in result["files"].values():
        path = out / entry["path"]
        data = path.read_bytes()
        if len(data) != entry["bytes"] or sha256_bytes(data) != entry["sha256"]:
            raise RuntimeError(f"bundle file {path.name} does not match its digest in result.json")
    forecast = pd.read_csv(out / result["files"]["forecast_csv"]["path"], parse_dates=["timestamp"])
    return forecast, result


def check_reload_parity(original: pd.DataFrame, reloaded: pd.DataFrame, *, tolerance: float = 1e-8) -> dict[str, Any]:
    """Compare the reloaded forecast with the in-memory one; raise on any mismatch (contract integrity)."""
    numeric = [c for c in original.columns if pd.api.types.is_numeric_dtype(original[c])]
    problems = []
    if list(original.columns) != list(reloaded.columns):
        problems.append(f"columns differ: {list(original.columns)} vs {list(reloaded.columns)}")
    elif len(original) != len(reloaded):
        problems.append(f"row counts differ: {len(original)} vs {len(reloaded)}")
    else:
        for c in original.columns:
            if c in numeric:
                delta = np.max(np.abs(original[c].to_numpy(float) - reloaded[c].to_numpy(float)))
                if delta > tolerance:
                    problems.append(f"{c}: max abs difference {delta:.3g} > {tolerance}")
            else:
                a = original[c].astype(str).to_numpy()
                b = reloaded[c].astype(str).to_numpy()
                if c == "timestamp":
                    a = pd.to_datetime(original[c]).astype(str).to_numpy()
                    b = pd.to_datetime(reloaded[c]).astype(str).to_numpy()
                if not np.array_equal(a, b):
                    problems.append(f"{c}: values differ after reload")
    parity = {
        "rows": int(len(original)),
        "columns": list(original.columns),
        "tolerance": tolerance,
        "ok": not problems,
        "problems": problems,
        "boundary": "forecast re-read from the CSV named in result.json, after a digest check",
    }
    if problems:
        raise RuntimeError("reload parity failed: " + "; ".join(problems))
    return parity

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `43046b85ec22…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `load_pinned_model(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

**Weights licence — read before running the next cell.** The cell downloads `google/timesfm-3.0-pytorch`, whose weights Google LLC distributes under the *TimesFM Non-Commercial License v1.0*: **non-commercial and non-production use only** (testing, evaluation and research not tied to commercial gain, production deployment or revenue generation), no use in end-user-facing or production systems, no redistribution of the model or of derivatives, and a commercial licence required for anything else. Downloading the weights means accepting those terms; the run records them in `resolved-revision.json` and in the exported provenance. The code in this notebook is Apache-2.0 and the sample is CC BY 4.0.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "timesfm-3.0-pytorch",
  "modelId": "google/timesfm-3.0-pytorch",
  "revision": "43046b85ec22d584a13f8098c2ed39c889e129c2",
  "revisionStatus": "pinned: the commit the Hub served for google/timesfm-3.0-pytorch at ref main on the maintainer's hosted Colab run of 2026-10-06 (resolved_revision in docs/execution-evidence/2026-10-06/timesfm_forecasting_colab_86f2213_run2_passed.ipynb)",
  "license": {
    "hub_front_matter": "license: other; license_name: timesfm-non-commercial-license-v1.0; license_link: LICENSE",
    "name": "TimesFM Non-Commercial License v1.0",
    "terms": "non-commercial and non-production use only; no distribution of the model or of derivatives; a commercial licence from Google LLC is required for any other use",
    "file": "LICENSE",
    "fileBytes": 7270,
    "url": "https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE"
  },
  "files": [
    {
      "path": "config.json",
      "bytes": 1273,
      "sha256": "ff17bbc07b792c5a904cca265b8468579d736a4fe84981da25eb871b0a125bc6"
    },
    {
      "path": "model.safetensors",
      "bytes": 1322898824,
      "sha256": "a7592b0a8432baee54483254e5647856911ce69e09d09a9bb65904b2d98f17da"
    }
  ],
  "totalBytes": 1322900097,
  "provenance": {
    "revision": "observed on the maintainer's hosted Colab run of 2026-10-06: the commit the Hub served for ref main (resolved_revision written by stage_missing_files to resolved-revision.json and exported in the run's provenance); evidence docs/execution-evidence/2026-10-06/timesfm_forecasting_colab_86f2213_run2_passed.ipynb, recorded in docs/release-verification.md",
    "config.json": "SHA-256 computed from the full Hub-served text (1,273 bytes, no trailing newline; the byte count equals the Hub listing) read through the Hugging Face connector on 2026-10-06",
    "model.safetensors": "SHA-256 computed in-run by verify_snapshot over the 1,322,898,824-byte file staged from the Hub on the maintainer's hosted Colab run of 2026-10-06 (weights_sha256 in the run's provenance; the manifest then held a pending sentinel, so the run reported it as not verified against the manifest); evidence docs/execution-evidence/2026-10-06/timesfm_forecasting_colab_86f2213_run2_passed.ipynb. Byte size verified against the Hub listing and the staged file",
    "fileListing": "file names and byte sizes of the checkpoint repository read through the Hugging Face connector on 2026-10-06: .gitattributes (1519), LICENSE (7270), README.md (1436), config.json (1273), model.safetensors (1322898824, LFS); model card front matter license: other, license_name: timesfm-non-commercial-license-v1.0, license_link: LICENSE, the LICENSE text read in full; Hub overview 330.7M parameters, last updated 2026-09-02",
    "architecture": "from config.json: input_patch_len 32, output_patch_len 64, nine quantiles 0.1..0.9, 20 transformer layers, model_dims 1280, 16 heads, variate attention, iterative CPM RevIN, linear detrending, stitching",
    "reVerification": "the 86f2213 run predates this pin; the next hosted run on the pinned head re-verifies the digest through verify_snapshot's hard check and the commit through resolved-revision.json"
  }
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = load_pinned_model(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Obtain and inspect the sample (or bring your own CSV)

*Core concept.* **Input:** one CSV, long format: a timestamp column, a numeric value column and, optionally, a series-id column. **System:** a pinned download with a size and SHA-256 check, or your own file. **Output:** a raw table the next section validates.

The default sample is **real weather**: hourly air temperature at 2 m for Manila, Cebu and Davao over 14 days (2026-08-26 to 2026-09-08 UTC), 3 series × 336 hours, from the Open-Meteo historical API (CC BY 4.0, *weather data by Open-Meteo.com*), fetched from a commit-pinned GitHub URL (this repository's own committed copy) and refused if its size or digest differs. The checkpoint's model card names its pretraining sources (GiftEvalPretrain, Wikipedia pageviews to November 2023, Google Trends to the end of 2022, synthetic and augmented data), and the observation window lies after every stated cut-off; the Hub revision of the checkpoint (2026-09-02) does overlap the window, so these exact values are unlikely, not impossible, to be in its pretraining data, and weather from the same sources for earlier periods may be. Three grid cells over two weeks are **tutorial data, not a benchmark**.

**Form fields.** `USE_BYOD`, `BYOD_PATH` and the three column fields (`TIMESTAMP_FIELD`, `VALUE_FIELD`, `SERIES_ID_FIELD`; leave the last empty for a single-series file) are the only values meant to be edited. With `USE_BYOD = True`, the cell reads `BYOD_PATH` when it is set and otherwise opens the Colab upload dialog (only then is `google.colab` imported). `HORIZON` is the number of steps held out and forecast; `CONTEXT_LENGTH` the most recent points shown to the model; `SEASON_LENGTH` the period of the seasonal-naive baseline and of MASE scaling (24 for hourly data with a daily cycle; set it to your data's period, or 1 to disable the seasonal baseline). `NAN_POLICY` and `GAP_POLICY` decide whether missing values and skipped timestamps are refused or filled with a report.

**Before you bring your own data**, read the data contract in the Prerequisites: your file needs at least `HORIZON + 16` rows per series, one sampling interval (or a calendar interval such as monthly), no duplicate timestamps, finite numeric values, and no more than 1,000 series or 2,000,000 rows. The next section names the file and the rule in every refusal. Do not upload data you are not authorised to process in a hosted runtime; the file stays in this runtime, and the notebook publishes nothing.

**What to notice.** The printed source record: the sample's URL, byte size and SHA-256, or your file's path; then the first rows and the per-series row counts.

In [ ]:
# Learner-facing: data source and request settings (form fields are the interface for executors too)
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
TIMESTAMP_FIELD = 'timestamp'  # @param {type:"string"}
VALUE_FIELD = 'target'  # @param {type:"string"}
SERIES_ID_FIELD = 'series_id'  # @param {type:"string"}
HORIZON = 24  # @param {type:"integer"}
CONTEXT_LENGTH = 312  # @param {type:"integer"}
SEASON_LENGTH = 24  # @param {type:"integer"}
NAN_POLICY = 'refuse'  # @param ["refuse", "interpolate"]
GAP_POLICY = 'refuse'  # @param ["refuse", "fill"]

import hashlib
import json
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

SAMPLE_URL = 'https://raw.githubusercontent.com/kurtvalcorza/timesfm-forecasting-pipeline/e47259ae75b93e7611c67e13a14d6607997c38f3/examples/sample-data/openmeteo_ph_hourly_temperature.csv'
SAMPLE_SHA256 = '74163ee609cda87869b7c13f4c2aa59343f94b4ba42d2e57331034902fe04f1a'
SAMPLE_BYTES = 31275
SAMPLE_LICENSE = 'CC BY 4.0 (weather data by Open-Meteo.com); the committed copy under examples/sample-data/ of kurtvalcorza/timesfm-forecasting-pipeline at a pinned commit'
Path('outputs').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)

config = ForecastConfig(
    horizon=int(HORIZON),
    context_length=int(CONTEXT_LENGTH),
    quantile_levels=(0.1, 0.5, 0.9),
    timestamp_column=TIMESTAMP_FIELD.strip(),
    value_column=VALUE_FIELD.strip(),
    series_id_column=SERIES_ID_FIELD.strip() or None,
    season_length=int(SEASON_LENGTH) if int(SEASON_LENGTH) > 1 else None,
    nan_policy=NAN_POLICY,
    gap_policy=GAP_POLICY,
)


def fetch_sample(url, sha256, n_bytes, cache_dir='data'):
    """Download the pinned sample once; refuse a size or digest mismatch before anything reads it."""
    target = Path(cache_dir) / url.rsplit('/', 1)[-1]
    if target.is_file() and target.stat().st_size == n_bytes and hashlib.sha256(target.read_bytes()).hexdigest() == sha256:
        return target, False
    with urllib.request.urlopen(url, timeout=60) as response:
        payload = response.read(n_bytes + 1)
    if len(payload) != n_bytes or hashlib.sha256(payload).hexdigest() != sha256:
        raise RuntimeError(
            f'The sample download does not match its pinned size/SHA-256 ({len(payload)} bytes); refusing to use it. '
            'Run the cell again; if it repeats, the file at the pinned URL was altered.'
        )
    target.write_bytes(payload)
    return target, True


if USE_BYOD:
    if BYOD_PATH.strip():
        byod_path = Path(BYOD_PATH.strip())
        if not byod_path.is_file():
            raise FileNotFoundError(f'BYOD_PATH {byod_path} is not a file; fix the path or clear it to use the upload dialog.')
    else:
        try:
            from google.colab import files  # imported only on this branch
        except ImportError as exc:
            raise RuntimeError('No upload dialog outside Colab: set BYOD_PATH to a CSV on this runtime.') from exc
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError(f'Upload exactly one CSV (got {len(uploaded)}), or set BYOD_PATH.')
        name, payload = next(iter(uploaded.items()))
        byod_path = Path('data') / Path(name).name
        byod_path.write_bytes(payload)
    data_source = {'kind': 'byod', 'path': str(byod_path), 'bytes': byod_path.stat().st_size, 'sha256': hashlib.sha256(byod_path.read_bytes()).hexdigest(), 'license': 'user-supplied'}
else:
    sample_path, downloaded = fetch_sample(SAMPLE_URL, SAMPLE_SHA256, SAMPLE_BYTES)
    data_source = {'kind': 'sample', 'name': 'Open-Meteo hourly 2 m temperature, Manila/Cebu/Davao, 2026-08-26..2026-09-08 UTC', 'path': str(sample_path), 'url': SAMPLE_URL, 'bytes': SAMPLE_BYTES, 'sha256': SAMPLE_SHA256, 'downloaded_this_run': downloaded, 'license': SAMPLE_LICENSE}

frame = load_series_csv(data_source['path'], config)
print({k: v for k, v in data_source.items() if k != 'url'})
print(frame.head(5).to_string(index=False))
print({'rows': len(frame), 'rows_per_series': frame[config.series_id_column].value_counts().to_dict() if config.series_id_column else {'series': len(frame)}})

## 5. Validate before the model runs

*Evaluation / engineering practice.* Validation is a first-class stage, not a side effect: `validate_series` parses the timestamps, infers the sampling interval (one fixed step, or a calendar interval such as monthly), finds skipped grid points and missing values, applies `NAN_POLICY` / `GAP_POLICY` while **reporting every change**, enforces the length ceilings, and returns a normalised table (`series_id`, `timestamp`, `value`) plus an **input manifest** that is written to `outputs/`. The model has not run yet.

Every refusal reads `[RULE] file: reason` and names the series, row or column, so a wrong file fails here and not inside the model. To show that, the second half of the cell builds a deliberately broken copy of the first series (one duplicated timestamp) and asks validation to refuse it; the printed line is what a BYOD mistake looks like. The check that the refusal *is* a `ValidationError` is a contract check: the notebook stops if validation ever lets a duplicate timestamp through.

**Prediction.** Before running: which rule do you expect to be the most common reason a real-world CSV is refused — a missing column, a gap, or too few rows?

**What to notice.** The inferred frequency (`h` for the sample), the per-series `n_rows_out`, an empty `changes` list for the sample (nothing was imputed or truncated), and the `[TIMESTAMP_DUPLICATE]` refusal for the probe.

In [ ]:
validation = validate_series(frame, config, source=data_source['path'])
input_manifest = validation.manifest()
with open('outputs/timesfm_forecasting_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2)
print({'frequency': input_manifest['frequency'], 'n_series': input_manifest['n_series'], 'n_rows': input_manifest['n_rows'], 'changes': input_manifest['changes']})
for report in input_manifest['series']:
    print({k: report[k] for k in ('series_id', 'n_rows_in', 'n_rows_out', 'start', 'end', 'n_interpolated', 'n_gap_points_filled', 'truncated_from')})
print({'ceilings': input_manifest['ceilings']})
print({'rules': list(RULES)})

# Refusal probe: a copy of the first series with one duplicated timestamp must be refused by name.
first_id = validation.series_ids[0]
probe = validation.frame[validation.frame['series_id'] == first_id].copy()
probe = pd.concat([probe, probe.iloc[[5]]], ignore_index=True)
probe_config = ForecastConfig(horizon=config.horizon, context_length=config.context_length, timestamp_column='timestamp', value_column='value', series_id_column='series_id', season_length=config.season_length)
try:
    validate_series(probe, probe_config, source='probe-duplicate-timestamp.csv')
except ValidationError as exc:
    refusal = {'code': exc.code, 'source': exc.source, 'message': str(exc)}
else:
    raise RuntimeError('validation accepted a duplicated timestamp; the data contract is broken')
print({'refusal_probe': refusal})

## 6. Split chronologically: the future is held out

*Core concept.* A forecast is only evidence if the model never saw the period it is scored on. `chronological_holdout` removes the **last `HORIZON` points of every series** as truth; everything before them is the history the baselines and the model may use. There is no random split in this package: shuffling a time series would let future values leak into the context.

**Prediction.** For the sample, the held-out day is 2026-09-08 UTC. Do you expect the three cities' temperature cycles on that day to look like the previous days (a daily cycle), or to contain something a history-only forecaster cannot know?

**What to notice.** The leakage check prints `True` per series — the last history timestamp is strictly before the first truth timestamp — and the notebook stops if it does not (a contract check, not a quality check).

In [ ]:
split = chronological_holdout(validation.frame, config.horizon)
leakage = split.leakage_check()
history_validation = validation.with_frame(split.history)
print({'history_rows': len(split.history), 'truth_rows': len(split.truth), 'horizon': split.horizon, 'leakage_check': leakage})
for series_id, g in split.truth.groupby('series_id', sort=False):
    print({'series_id': series_id, 'truth_from': str(g['timestamp'].min()), 'truth_to': str(g['timestamp'].max()), 'history_to': str(split.history[split.history['series_id'] == series_id]['timestamp'].max())})
if not all(leakage.values()):
    raise RuntimeError(f'holdout leakage check failed: {leakage}; the split is not chronological')

## 7. Baselines first: what does "good" have to beat?

*Evaluation practice.* Three history-only forecasters set the bar before the model runs:

- **naive** — repeat the last observed value;
- **seasonal naive** — repeat the last full season (`SEASON_LENGTH` steps), the usual rule of thumb for data with a daily cycle;
- **SES** — simple exponential smoothing: a flat forecast at a level that weights recent points more, with the smoothing weight chosen on the history alone.

Each is scored with the same three metrics the model will get: **MASE** (mean absolute error divided by the in-sample seasonal-naive error; below 1 beats that rule on the history's own scale, and it is unit-free so series can be pooled), **sMAPE** (symmetric percentage error) and, for the model only, the **quantile loss** (pinball loss) at each exported quantile.

**Prediction.** Rank the three baselines before you run the cell. For hourly temperature with a clear daily cycle, which one should have the lowest MASE, and why can the naive forecast be badly wrong at a 24-step horizon?

**What to notice.** The table's `mase` column. A seasonal-naive MASE well below the naive one says the daily cycle is strong; a value near or below 1 says the last day was a good template for the next.

In [ ]:
frequency = validation.frequency
baselines = {'naive': naive_baseline(split.history, frequency, config.horizon), 'ses': ses_baseline(split.history, frequency, config.horizon)}
if config.season_length is not None:
    baselines['seasonal_naive'] = seasonal_naive_baseline(split.history, frequency, config.horizon, config.season_length)
baseline_evals = {name: evaluate_forecast(frame_, split.truth, split.history, config) for name, frame_ in baselines.items()}
baseline_table = pd.DataFrame([{'method': name, **{k: round(ev['aggregate'][k], 4) for k in ('mae', 'mase', 'smape')}} for name, ev in baseline_evals.items()])
print(baseline_table.to_string(index=False))
print({'mase_scale': f"in-sample seasonal-naive MAE with season_length={config.season_length or 1}", 'holdout': 'last %d steps per series' % config.horizon})

## 8. Zero-shot forecast with TimesFM

*Core concept.* **Input:** the most recent `CONTEXT_LENGTH` points of each series, as plain numbers (TimesFM 3.0 needs no frequency label). **Model:** a stacked mixing transformer (20 layers, model dimension 1280, 16 heads; the Hub lists 330.7M parameters) pretrained on a large corpus of time series, reading 32-point input patches and decoding 64-point output patches; nothing is trained or tuned here, which is what *zero-shot* means. **Output:** for every series and future step, the **median** forecast (`prediction`, the model's quantile 0.5, slot 4 of its nine quantile slots) and the requested **quantiles** (`q0.1`, `q0.5`, `q0.9`). TimesFM 3.0 has no separate mean head, so none is exported.

**Quantiles are model quantiles.** The band between `q0.1` and `q0.9` is what the model's quantile head outputs for the nominal 80 % interval (sorted monotone by the upstream forecaster); it is **not** a calibrated prediction interval for this data until Section 9 measures how often the truth actually falls inside it. The point forecast is the median: with a skewed quantile spread it is not the centre of the band.

**Prediction.** Will the model's first few steps hug the last observed value (like the naive forecast) or follow the daily cycle (like the seasonal baseline)? Will the band widen with the horizon?

**What to notice.** The plot: history tail, truth (dashed), median forecast and the shaded q0.1–q0.9 band for the first series. Look at where the truth leaves the band.

In [ ]:
import matplotlib.pyplot as plt

result = forecast(history_validation, config, pipe)
forecast_frame = result.forecast
print(forecast_frame.head(6).to_string(index=False))
print({'effective_context': result.provenance['effective_context_length'], 'decode_settings': result.provenance['decode_settings'], 'point_semantics': result.provenance['point_forecast_semantics']})

plot_id = validation.series_ids[0]
hist_tail = split.history[split.history['series_id'] == plot_id].tail(3 * config.horizon)
truth_part = split.truth[split.truth['series_id'] == plot_id]
fc_part = forecast_frame[forecast_frame['series_id'] == plot_id]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_tail['timestamp'], hist_tail['value'], color='0.3', label='history (tail)')
ax.plot(truth_part['timestamp'], truth_part['value'], color='black', linestyle='--', label='held-out truth')
ax.plot(fc_part['timestamp'], fc_part['prediction'], color='tab:blue', label='TimesFM median')
ax.fill_between(fc_part['timestamp'], fc_part['q0.1'], fc_part['q0.9'], color='tab:blue', alpha=0.2, label='q0.1-q0.9 (model quantiles)')
if 'seasonal_naive' in baselines:
    sn_part = baselines['seasonal_naive'][baselines['seasonal_naive']['series_id'] == plot_id]
    ax.plot(sn_part['timestamp'], sn_part['prediction'], color='tab:orange', linestyle=':', label='seasonal naive')
ax.set_title(f'{plot_id}: {config.horizon}-step zero-shot forecast vs held-out truth')
ax.legend(loc='upper left', fontsize=8)
fig.autofmt_xdate()
plt.show()

## 9. Evaluate beside the baselines (verdicts are recorded, not asserted)

*Evaluation practice.* The model's forecast is scored on the **same held-out rows with the same metrics** as the baselines, and the comparison is written to `outputs/` as an evaluation report. The report's verdict is `sample-sanity`: one chronological holdout on one small sample shows that the pipeline works end to end and how the model compares *on this sample*. It is not a benchmark, and the notebook does not stop if the model loses to a baseline — a foundation model losing to a seasonal rule on a strongly periodic two-week series is a legitimate, informative result.

Two more numbers belong to the model alone: the mean **quantile loss** over q0.1 / q0.5 / q0.9 (lower is better; the q0.5 term is half the MAE), and the empirical **coverage** of the q0.1–q0.9 band — the share of truth points inside it, to compare with the nominal 0.8.

**What to notice.** Whether `timesfm` has a lower MASE than `seasonal_naive`, and the coverage: well below 0.8 means the band is too narrow for this data; well above means too wide.

<details><summary>Check your reasoning</summary>

The seasonal-naive baseline encodes exactly the structure hourly temperature has — a daily cycle — so it is the one to watch. If the model beats it on MASE, the model has used more than the last day (several days of cycle, plus the trend between them). If it does not, remember what the holdout is: one day per city. A single unusual day (rain, a front) moves every method, and the ranking can flip on another day. Coverage below the nominal 0.8 is the common outcome for a zero-shot band on a new domain: read it as *the band is a model statement, not a guarantee*. Only a longer backtest over many origins could say whether the ranking is stable.

</details>

In [ ]:
model_eval = evaluate_forecast(forecast_frame, split.truth, split.history, config)
sample_kind = data_source.get('name', 'user-supplied data') if data_source['kind'] == 'sample' else 'user-supplied data (BYOD)'
report = evaluation_report(model_eval, baseline_evals, horizon=config.horizon, sample_kind=sample_kind)
with open('outputs/timesfm_forecasting_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2)
comparison_table = pd.DataFrame(report['table'])
print(comparison_table.round(4).to_string(index=False))
print({'verdict': report['verdict'], 'reason': report['reason']})
print({'model_beats_on_mase': report['model_beats_on_mase'], 'quantile_loss': {k: round(v, 4) for k, v in report['model_quantile_loss'].items()}, 'interval': report['interval'], 'interval_coverage': round(report['model_interval_coverage'], 3) if report['model_interval_coverage'] is not None else None})
for series_id, entry in model_eval['per_series'].items():
    print({'series_id': series_id, 'mase': round(entry['mase'], 4), 'smape': round(entry['smape'], 3), 'coverage': round(entry.get('interval_coverage', float('nan')), 3)})

## 10. Activity: Predict → Change one thing → Run → Observe → Explain

*Evaluation practice (optional; bounded; it does not change the canonical results above).* The one thing that changes is the **context length**: `ACTIVITY_CONTEXT_LENGTH` points of history instead of `CONTEXT_LENGTH`. Everything else — the split, the truth, the metrics, the baselines — stays fixed, so any difference in the table is caused by how much history the model sees.

1. **Predict.** With a short context (the default below is two days of hourly data), will the model's MASE get worse, better, or stay about the same? Will the q0.1–q0.9 band get wider or narrower?
2. **Change one thing.** Edit `ACTIVITY_CONTEXT_LENGTH` if you like (any value from 16 up to the series length); the default is a bounded, safe choice.
3. **Run** the cell.
4. **Observe** the two rows of the printed table.
5. **Explain** the difference in one sentence, in terms of what a shorter context removes (earlier days of the cycle, the week-scale trend) and what it keeps (the most recent day).

<details><summary>Check your reasoning</summary>

With a daily cycle, two days of context still contain the pattern, so the median often stays close; what usually changes is the spread, because the model has fewer repetitions of the cycle to judge its regularity from. A larger loss of accuracy with a short context points to information beyond the last two days — a slow drift between days — that the longer context carried. There is no universally right context length; this is why the pipeline reports the *effective* context in provenance rather than assuming one.

</details>

In [ ]:
ACTIVITY_CONTEXT_LENGTH = 48  # @param {type:"integer"}

activity_config = ForecastConfig(
    horizon=config.horizon, context_length=int(ACTIVITY_CONTEXT_LENGTH), quantile_levels=config.quantile_levels,
    timestamp_column=config.timestamp_column, value_column=config.value_column, series_id_column=config.series_id_column,
    season_length=config.season_length, nan_policy=config.nan_policy, gap_policy=config.gap_policy,
)
activity_result = forecast(history_validation, activity_config, pipe)
activity_eval = evaluate_forecast(activity_result.forecast, split.truth, split.history, activity_config)
width = lambda f: float((f['q0.9'] - f['q0.1']).mean())  # noqa: E731 - one-line helper for the printed table
activity_table = pd.DataFrame([
    {'context_length': config.context_length, 'effective': result.provenance['effective_context_length'], 'mase': round(model_eval['aggregate']['mase'], 4), 'smape': round(model_eval['aggregate']['smape'], 3), 'mean_band_width': round(width(forecast_frame), 3), 'coverage': model_eval['aggregate'].get('interval_coverage')},
    {'context_length': activity_config.context_length, 'effective': activity_result.provenance['effective_context_length'], 'mase': round(activity_eval['aggregate']['mase'], 4), 'smape': round(activity_eval['aggregate']['smape'], 3), 'mean_band_width': round(width(activity_result.forecast), 3), 'coverage': activity_eval['aggregate'].get('interval_coverage')},
])
print(activity_table.to_string(index=False))
activity_record = {'changed': 'context_length', 'values': [config.context_length, activity_config.context_length], 'mase': [model_eval['aggregate']['mase'], activity_eval['aggregate']['mase']], 'mean_band_width': [width(forecast_frame), width(activity_result.forecast)], 'verdict': 'recorded; no quality assertion'}
print(activity_record)

## 11. Forecast beyond the end of the data

*Core concept.* The evaluation above forecast a period whose truth we had. The operational use is the other one: give the model **all** the history and forecast the next `HORIZON` steps, for which no truth exists yet. The output has the same columns as before and is written to `outputs/`; its evaluation verdict is `not-measurable`, and the notebook says so rather than inventing a score.

**What to notice.** The timestamps continue the series' grid (the next hours after 2026-09-08 23:00 UTC for the sample), and the first forecast values start near the last observed value.

In [ ]:
future_result = forecast(validation, config, pipe)
future_frame = future_result.forecast
future_frame.to_csv('outputs/timesfm_forecasting_future_forecast.csv', index=False, lineterminator='\n')
future_report = {'verdict': 'not-measurable', 'reason': 'no truth exists for these timestamps yet; score them with evaluate_forecast once it does', 'horizon': config.horizon, 'from': str(future_frame['timestamp'].min()), 'to': str(future_frame['timestamp'].max())}
print(future_frame.groupby('series_id', sort=False).head(2).to_string(index=False))
print(future_report)

## 12. Export the result bundle and reload it

*Engineering / reproducibility.* `write_result_bundle` writes `outputs/timesfm_forecasting_forecast.csv` (the held-out forecast with its quantiles), `timesfm_forecasting_evaluation.json`, `timesfm_forecasting_provenance.json` (model identity, licence terms and revision status, the observed weight digest, the runtime versions, the configuration, the input manifest, the data source) and `timesfm_forecasting_result.json`, which indexes the other files with their byte sizes and SHA-256 digests and carries the evaluation report. `reload_result_bundle` then reads the forecast **back from the files** after checking those digests, and `check_reload_parity` compares it with the in-memory forecast. Parity is a contract check — it proves the exported bundle reproduces what was evaluated, not that the forecast is good — so a mismatch stops the notebook.

**What to notice.** `reload_parity.ok` is `True`, and `sorted(os.listdir('outputs'))` lists every file this notebook produced.

In [ ]:
provenance = build_provenance(pipe, config, history_validation, result, data_source=data_source, notebook_source=NOTEBOOK_SOURCE)
provenance['holdout'] = {'estimation': 'chronological tail holdout', 'horizon': config.horizon, 'leakage_check': leakage}
provenance['future_forecast'] = future_report
bundle_paths = write_result_bundle(
    'outputs', 'timesfm_forecasting', forecast=forecast_frame, evaluation=model_eval, report=report, provenance=provenance,
    extra={'baselines': {name: ev['aggregate'] for name, ev in baseline_evals.items()}, 'activity': activity_record, 'input_manifest_file': 'timesfm_forecasting_input_manifest.json', 'future_forecast_file': 'timesfm_forecasting_future_forecast.csv'},
)
reloaded_forecast, reloaded_result = reload_result_bundle('outputs', 'timesfm_forecasting')
reload_parity = check_reload_parity(forecast_frame, reloaded_forecast)
print({'bundle': bundle_paths, 'reload_parity': {k: reload_parity[k] for k in ('rows', 'ok', 'tolerance', 'boundary')}})
print({'model': {k: provenance['model'].get(k) for k in ('model_id', 'revision', 'resolved_revision', 'weights_sha256', 'weights_sha256_verified_against_manifest', 'license', 'license_terms')}, 'device': provenance['device'], 'runtime': {k: provenance['runtime'].get(k) for k in ('python', 'timesfm', 'torch', 'cuda_available')}})
print(sorted(os.listdir('outputs')))

## Interpretation and limits

**Conclude with evidence.** Write a short conclusion from your own run using this scaffold (an optional personal note, not a required submission):

> On [data, series, holdout], the zero-shot TimesFM median forecast reached MASE [value] and sMAPE [value], against [value] for the seasonal-naive baseline and [value] for the naive forecast; the q0.1–q0.9 band covered [share] of the held-out points against a nominal 0.8. Shortening the context to [value] points changed MASE by [amount]. This suggests [a bounded conclusion about this sample], but does not establish [a broader claim about the model or about other series].

**What this evidence supports.** On the demonstrated sample the pipeline validated a real series into an input manifest, held out the future chronologically, scored three history-only baselines, produced a zero-shot median-and-quantile forecast from the pinned checkpoint, scored it with the same metrics, varied one thing (the context) in a controlled way, forecast beyond the data, and exported a bundle that reloads with parity. Whatever the ranking in your run, that ranking is the evidence — recorded, not asserted.

**What it does not establish.** One holdout day per series on three grid cells over two weeks cannot rank forecasting models or show that a ranking is stable; the quantile band is the model's statement and its measured coverage on one day is not a calibration result; MASE and sMAPE do not characterise every error mode (a forecast that misses the timing of the daily peak can score well on sMAPE); the pipeline passes no covariates to TimesFM 3.0 (the upstream covariate inputs are not exposed on this path), so weather drivers, holidays or interventions are invisible to it; and nothing here shows how the model behaves on series with structural breaks, on counts near zero, or at horizons far beyond the sample's daily cycle. The checkpoint's pretraining mixture is described upstream and cannot be reconstructed here; weather from the same sources for earlier dates may be in it.

**Carry to real data.** Read the seasonal-naive and naive rows on *your* series before the model's; if the seasonal rule wins, say so. Use a backtest over many forecast origins (rolling holdouts) before trusting a ranking, measure the band's coverage on that backtest before using it to size anything, and keep the horizon within what your decision needs. Set `SEASON_LENGTH` to your data's real period (7 for daily data with a weekly cycle, 12 for monthly data with a yearly cycle) or the MASE scale and the seasonal baseline will be wrong for it.

**Next experiments (optional; they do not affect the default path).** Note the numbers you want to compare before changing anything; in Colab, **Runtime → Run after** runs the selected cell and every cell below it.

- **Another horizon:** set `HORIZON = 48` in Section 4 and choose **Run after**: the holdout becomes two days, the seasonal baseline repeats one day twice, and the band should widen with the step.
- **Fill instead of refuse:** delete a few rows from a copy of the sample, set `GAP_POLICY = 'fill'` and `NAN_POLICY = 'interpolate'` with `USE_BYOD = True` and `BYOD_PATH` pointing to the copy, and read the `changes` list in Section 5 before the metrics.
- **Monthly data:** bring a monthly series (`SEASON_LENGTH = 12`); validation reports the calendar interval `MS`, and the baselines and the model use it for the future timestamps.
- **Your own data:** BYOD (Section 4, then **Run after**); read the baselines before the model's number.

**Bring your own data.** Supply one CSV with a timestamp column, a numeric value column and optionally a series-id column; set the three column fields in Section 4. The same validation, split, baselines, forecast, evaluation and export run on it; a refusal names the file and the rule. Only upload data you are permitted to process in a hosted runtime; files stay in this runtime and are not published by the notebook.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can build its isolated environment, stage the pinned checkpoint at its immutable commit and check its sizes and digests, fetch and digest-verify a real public series, validate the data contract, split without leakage, run the zero-shot forecast with point and quantile output, score it beside history-only baselines, and emit the shown machine-readable bundle with reload parity — without the repository being reachable. It does **not** establish benchmark superiority, calibration, forecast quality on other series or horizons, or production fitness.

**AI Assistance Disclosure:** This notebook's code and explanations were developed with generative AI assistance under maintainer direction. The maintainer remains responsible for reviewing the implementation, validating results and making release decisions. AI assistance does not constitute independent verification, provider endorsement or release approval.

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment is built from manylinux wheels. Use Google Colab, Kaggle or a Linux Jupyter server; Windows and macOS kernels are not supported.
- **Section 1 fails while downloading** (`uv` wheel, Python build or packages). The runtime needs PyPI and the python-build-standalone download; run the cell again once the network is back. A "size/SHA-256" or "does not match its digest" error means a file was altered: do not edit the cell, regenerate the notebook from the repository.
- **"The isolated environment's Python process exited".** Usually the runtime ran out of memory. Restart the session and choose **Run all**; on CPU, close other notebooks first. The checkpoint needs about 1.3 GB of RAM in float32 plus the context.
- **Out of disk or the checkpoint download stops.** The locked install (PyTorch with its CUDA libraries) and the 1.32 GB checkpoint need several GB of free space. Start a fresh runtime, or delete `dimer_isolated_env_*/` and `weights/` from an earlier attempt.
- **A size or digest mismatch in Section 3 or 4.** A download was incomplete or altered; the notebook refuses it rather than continuing. Run the cell again; never edit the manifest or the sample digest. A `config.json` mismatch means the Hub served a different checkpoint than the pinned one — stop and report it.
- **Section 3 reports a `model.safetensors` SHA-256 mismatch, or a commit other than the pinned one in `resolved-revision.json`.** The Hub served bytes or a commit that differ from what the maintainer's hosted run of 2026-10-06 observed and pinned; the notebook refuses the snapshot. Run the cell again on a fresh runtime; if it repeats, stop and report it with the printed digest and commit.
- **BYOD is refused in Section 5.** The message names the file and the rule: `[REQUIRED_COLUMNS]` (set the column fields), `[TIMESTAMP_DUPLICATE]`, `[FREQUENCY_GAP]` / `[FREQUENCY_IRREGULAR]`, `[VALUE_MISSING]` / `[VALUE_NUMERIC]`, `[MIN_HISTORY]` (fewer than `HORIZON + 16` rows), `[FREQUENCY_MIXED]`. Fix the file or the policy fields and choose **Run after** from Section 4. An empty or cancelled upload asks you to choose a file or set `BYOD_PATH`.
- **`SEASON_TOO_LONG` in Section 7.** A series' history is shorter than `SEASON_LENGTH`; lower it (or set it to 1 to disable the seasonal baseline).
- **Reload parity fails in Section 12.** The exported bundle does not reproduce the evaluated forecast; this is a contract failure, not a data problem. Restart the session and choose **Run all**; if it repeats, report it with the printed problems.
- **Numbers differ slightly between CPU and GPU runs.** CUDA kernels are not bit-for-bit identical to CPU float32; differences in the last digits are expected. Larger differences on the default settings mean a form field was changed.
- **You re-ran the Section 1 cell.** It reuses the environment and keeps the live worker, so later cells keep working; only a restarted session starts over.

## Glossary

- **Zero-shot forecasting:** forecasting a series the model was not trained or tuned on, from its history alone.
- **Horizon:** the number of future steps forecast (`HORIZON`); also the length of the holdout.
- **Context (length):** the most recent history points the model reads; the *effective* context is what it actually received (shorter series are passed whole).
- **Chronological holdout:** removing the last `HORIZON` points of each series as truth so that the model never sees the period it is scored on.
- **Leakage:** any way future information reaches the forecaster; a random split would be one.
- **Naive / seasonal naive / SES:** repeat the last value; repeat the last season; a flat forecast at an exponentially-smoothed level.
- **Season length:** the period of the cycle the seasonal baseline repeats and MASE scales by (24 for hourly data with a daily cycle).
- **MASE:** mean absolute scaled error — MAE divided by the in-sample seasonal-naive MAE; unit-free; below 1 beats that rule on the history's scale.
- **sMAPE:** symmetric mean absolute percentage error, in percent.
- **Quantile (pinball) loss:** the loss that a correct quantile forecast minimises; lower is better.
- **Model quantiles / coverage:** the quantile head's outputs (q0.1 … q0.9) and the measured share of truth points inside a band; a model band is not a calibrated interval until coverage is measured on enough data.
- **Median as the point forecast:** the point forecast here is the median (q0.5, slot 4 of the model's nine quantile slots); TimesFM 3.0 has no mean head, and with a skewed spread the median is not the centre of the band.
- **Input manifest:** the machine-readable record of what was validated, under which rules, ceilings and policies, and what changed.
- **Result bundle / reload parity:** the exported CSV and JSON files indexed by digest in `result.json`, and the check that the bundle reproduces the in-memory forecast.

## References

- Das, A., Kong, W., Sen, R., & Zhou, Y. (2024). A decoder-only foundation model for time-series forecasting. *Proceedings of the 41st International Conference on Machine Learning (ICML 2024)*. https://arxiv.org/abs/2310.10688
- Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting, 22*(4), 679–688. https://doi.org/10.1016/j.ijforecast.2006.03.001
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: principles and practice* (3rd ed.). OTexts. https://otexts.com/fpp3/
- Google Research. (2026). TimesFM 3.0 PyTorch checkpoint (TimesFM Non-Commercial License v1.0). Hugging Face Hub. https://huggingface.co/google/timesfm-3.0-pytorch
- Open-Meteo. (2026). Historical weather API. https://open-meteo.com/ (data: CC BY 4.0)